In [ ]:
"""
========================================================
FULL COLAB BENCHMARK — Poly PCA+OLS vs OLS vs RF
Paste this ENTIRE cell into Google Colab and run it.
GPU recommended (T4 is fine, ~15-20 min total).
========================================================
"""

# !pip install -q scikit-learn pandas openpyxl xlrd  # uncomment if needed

import time, warnings, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing, load_diabetes, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
warnings.filterwarnings('ignore')

# ==========================================
# CONFIG
# ==========================================
DATASETS = {
    'Yacht':       True,    # 308 obs, 6 feat
    'Energy':      True,    # 768 obs, 8 feat
    'Concrete':    True,    # 1030 obs, 8 feat
    'Airfoil':     True,    # 1503 obs, 5 feat
    'Abalone':     True,    # 4177 obs, 8 feat
    'California':  True,    # 20640 obs, 8 feat
    'Kin8nm':      True,    # 8192 obs, 8 feat
    'Protein':     True,    # 45730 obs, 9 feat
}

N_REPEATS = 5
TEST_FRAC = 0.2
MAX_N = 5000
SEED = 42

# Models to run
RUN_OLS  = True
RUN_RF   = True
RUN_POLY = True

# RF config
RF_TREES = 500; RF_MTRY = 1/3

# Poly PCA+OLS config
POLY_PCA_COMP = 200
POLY_D_MODEL = 64
POLY_FFN_DIM = 128
POLY_EPOCHS = 50
POLY_BS = 256
POLY_LR = 1e-3
POLY_DROPOUT = 0.1
POLY_WD = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ==========================================
# DATASET LOADING
# ==========================================
OPENML_REGISTRY = {
    'Concrete':  (44959,  (1000, 1100),  8),
    'Energy':    (41478,  (700, 800),     8),
    'Kin8nm':    (189,    (8000, 8300),   8),
    'Protein':   (44963,  (45000, 46000), 9),
    'Yacht':     (42370,  (300, 320),     6),
    'Airfoil':   (44957,  (1400, 1600),   5),
    'Abalone':   (183,    (4100, 4200),   7),
}

def load_datasets():
    enabled = [k for k, v in DATASETS.items() if v]
    print(f"\nLoading {len(enabled)} datasets...\n")
    loaded = {}

    if 'California' in enabled:
        try:
            d = fetch_california_housing()
            loaded['California'] = (d.data, d.target)
            print(f"  California:  {d.data.shape}")
        except Exception as e:
            print(f"  California FAILED: {e}")

    for name in enabled:
        if name == 'California':
            continue
        if name not in OPENML_REGISTRY:
            continue
        data_id, (n_min, n_max), expected_p = OPENML_REGISTRY[name]
        try:
            data = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
            X = data.data.select_dtypes(include=[np.number]).values
            y_raw = data.target
            if hasattr(y_raw, 'shape') and len(y_raw.shape) > 1 and y_raw.shape[1] > 1:
                y_raw = y_raw.iloc[:, 0]
            elif hasattr(y_raw, 'columns'):
                y_raw = y_raw.iloc[:, 0]
            if hasattr(y_raw, 'cat') and y_raw.dtype.name == 'category':
                y = y_raw.cat.codes.values.astype(float)
            else:
                y = np.array(y_raw).astype(float)
            mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
            X, y = X[mask], y[mask]
            N, P = X.shape
            if not (n_min <= N <= n_max) or P != expected_p:
                print(f"  {name}: shape mismatch ({N},{P}), skipping")
                continue
            loaded[name] = (X, y)
            print(f"  {name}:{' '*(12-len(name))}{X.shape}")
        except Exception as e:
            print(f"  {name} FAILED: {e}")

    print(f"\n{len(loaded)} datasets ready.\n")
    return loaded


# ==========================================
# POLY PCA+OLS MODEL
# ==========================================
def train_poly_model(X_train, y_train, X_test,
                     n_components=200, d_model=64, ffn_dim=128,
                     epochs=50, batch_size=256, lr=1e-3,
                     dropout=0.1, weight_decay=1e-4,
                     seed=42, verbose=False):
    """
    Polynomial Cross-Feature Transformer with PCA compression + OLS warm-start.
    Replaces self-attention with explicit polynomial interaction features.
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]
    D = d_model

    def poly_cross(x):
        B, S, Dm = x.shape
        per = torch.cat([x, x**2 - 1], dim=-1)
        pairs = torch.cat([x[:, i, :] * x[:, j, :]
                           for i in range(S) for j in range(i, S)], dim=-1)
        return torch.cat([per, pairs.unsqueeze(1).expand(B, S, -1)], dim=-1)

    class FrozenPCA(nn.Module):
        def __init__(self, mean, components):
            super().__init__()
            self.register_buffer('mean', torch.from_numpy(mean.astype(np.float32)))
            self.register_buffer('comp', torch.from_numpy(components.astype(np.float32)))
            self.out_dim = components.shape[0]
        def forward(self, x):
            return (x - self.mean) @ self.comp.T

    class PolyPCAModel(nn.Module):
        def __init__(self, nf, pca_layer, d, ffn_dim, drop):
            super().__init__()
            self.pca = pca_layer
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            self.W = nn.Linear(pca_layer.out_dim, d)
            self.n1 = nn.LayerNorm(d)
            self.ffn = nn.Sequential(
                nn.Linear(d, ffn_dim), nn.ReLU(), nn.Dropout(drop),
                nn.Linear(ffn_dim, d), nn.Dropout(drop))
            self.n2 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)
        def embed_x(self, x):
            return self.embed(x.unsqueeze(-1)) + self.pos
        def get_poly(self, emb):
            return self.pca(poly_cross(emb))
        def forward(self, x):
            emb = self.embed_x(x)
            phi = self.get_poly(emb)
            z = self.n1(emb + self.W(phi))
            return self.head(self.n2(z + self.ffn(z)).mean(1))

    # Step 1: Fit PCA on training poly features
    torch.manual_seed(seed)
    tmp_embed = nn.Linear(1, D)
    tmp_pos = nn.Parameter(torch.randn(1, nf, D) * 0.02)
    Xt = torch.from_numpy(X_train.astype(np.float32))
    with torch.no_grad():
        emb = tmp_embed(Xt.unsqueeze(-1)) + tmp_pos
        phi_all = poly_cross(emb).reshape(-1, emb.shape[-1] * 2 + D * nf * (nf + 1) // 2)
    n_comp = min(n_components, phi_all.shape[1], phi_all.shape[0])
    pca = PCA(n_components=n_comp, random_state=seed)
    pca.fit(phi_all.numpy())
    pca_layer = FrozenPCA(pca.mean_, pca.components_)

    # Step 2: Build model
    torch.manual_seed(seed)
    model = PolyPCAModel(nf, pca_layer, D, ffn_dim, dropout).to(device)

    # Step 3: OLS warm-start
    model.eval()
    Xt_dev = Xt.to(device)
    with torch.no_grad():
        emb = model.embed_x(Xt_dev)
        phi_pooled = model.get_poly(emb).mean(1).cpu().numpy()
    reg = Ridge(alpha=1.0)
    reg.fit(phi_pooled, y_train)
    with torch.no_grad():
        model.W.weight.data[0] = torch.from_numpy(reg.coef_.astype(np.float32)).to(device)
        model.W.bias.data.zero_()
        model.head.weight.data.zero_()
        model.head.weight.data[0, 0] = 1.0
        model.head.bias.data.fill_(float(reg.intercept_))

    # Step 4: Train
    yt = torch.from_numpy(y_train.astype(np.float32)).unsqueeze(1)
    tr_loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()

    for ep in range(1, epochs + 1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()

    # Step 5: Predict
    model.eval()
    Xte = torch.from_numpy(X_test.astype(np.float32)).to(device)
    with torch.no_grad():
        y_pred = model(Xte).cpu().numpy().ravel()

    info = {
        'n_params': sum(p.numel() for p in model.parameters()),
        'pca_components': n_comp,
    }
    return y_pred, info


# ==========================================
# BENCHMARK
# ==========================================
def run_benchmark():
    datasets = load_datasets()
    if not datasets:
        print("No datasets loaded.")
        return None

    models_active = [m for m, on in [
        ('OLS', RUN_OLS), ('RF', RUN_RF), ('Poly', RUN_POLY)] if on]

    print(f"Models: {', '.join(models_active)}")
    print(f"Repeats: {N_REPEATS}, Test: {TEST_FRAC}, Max N: {MAX_N}\n")

    results = []
    timings = []
    t0_global = time.time()

    for ds_name, (X_full, y_full) in datasets.items():
        N_full, P = X_full.shape
        if N_full > MAX_N:
            np.random.seed(SEED)
            idx = np.random.choice(N_full, MAX_N, replace=False)
            X_use, y_use = X_full[idx], y_full[idx]
            N_use = MAX_N
        else:
            X_use, y_use = X_full, y_full
            N_use = N_full

        print(f"{'='*75}")
        print(f"  {ds_name} (N={N_use}, P={P})")
        print(f"{'='*75}")

        scores = {k: [] for k in models_active}
        times  = {k: [] for k in models_active}

        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            print(f"  Rep {r+1}/{N_REPEATS}...", end=" ", flush=True)

            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_train).astype(np.float32)
            X_te = scaler.transform(X_test).astype(np.float32)
            y_train = y_train.astype(np.float32)
            y_test = y_test.astype(np.float32)

            if 'OLS' in models_active:
                t0 = time.time()
                m = LinearRegression().fit(X_tr, y_train)
                scores['OLS'].append(r2_score(y_test, m.predict(X_te)))
                times['OLS'].append(time.time() - t0)

            if 'RF' in models_active:
                t0 = time.time()
                m = RandomForestRegressor(n_estimators=RF_TREES, max_features=RF_MTRY,
                                          n_jobs=-1, random_state=seed).fit(X_tr, y_train)
                scores['RF'].append(r2_score(y_test, m.predict(X_te)))
                times['RF'].append(time.time() - t0)

            if 'Poly' in models_active:
                t0 = time.time()
                try:
                    y_pred, info = train_poly_model(
                        X_tr, y_train, X_te,
                        n_components=POLY_PCA_COMP, d_model=POLY_D_MODEL,
                        ffn_dim=POLY_FFN_DIM, epochs=POLY_EPOCHS,
                        batch_size=POLY_BS, lr=POLY_LR,
                        dropout=POLY_DROPOUT, weight_decay=POLY_WD,
                        seed=seed, verbose=False)
                    scores['Poly'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[Poly:{e}]", end=" ")
                    scores['Poly'].append(np.nan)
                times['Poly'].append(time.time() - t0)

            print("done")

        # Report per dataset
        avg = {k: np.nanmean(v) for k, v in scores.items()}
        se  = {k: np.nanstd(v)  for k, v in scores.items()}
        avg_t = {k: np.mean(v) for k, v in times.items()}

        print(f"\n  {'Model':<8} {'R² (mean)':>10} {'± std':>8} {'Time/rep':>10}")
        print(f"  {'-'*40}")
        for m in models_active:
            print(f"  {m:<8} {avg[m]:>10.4f} {se[m]:>8.4f} {avg_t[m]:>9.1f}s")
        print()

        results.append({
            'Dataset': ds_name, 'N': N_use, 'P': P,
            **{m: avg[m] for m in models_active},
            **{f'{m}_se': se[m] for m in models_active},
            **{f'{m}_time': avg_t[m] for m in models_active},
        })

    elapsed = time.time() - t0_global

    # ── Grand summary ─────────────────────────────────────────────
    df = pd.DataFrame(results)

    print(f"\n{'='*90}")
    print(f"  RESULTS ({elapsed/60:.1f} min total, {N_REPEATS} reps, device={DEVICE})")
    print(f"{'='*90}")

    header = f"  {'Dataset':<12} {'N':>5} {'P':>3}"
    for m in models_active:
        header += f"  {'R²_'+m:>9}"
    for m in models_active:
        header += f"  {'t_'+m:>7}"
    print(header)
    print(f"  {'-'*85}")

    for _, row in df.iterrows():
        vals = {m: row[m] for m in models_active}
        best_val = max(vals.values())
        line = f"  {row['Dataset']:<12} {int(row['N']):>5} {int(row['P']):>3}"
        for m in models_active:
            v = row[m]
            marker = " *" if abs(v - best_val) < 0.001 else "  "
            line += f"  {v:>7.4f}{marker}"
        for m in models_active:
            line += f"  {row[f'{m}_time']:>6.1f}s"
        print(line)

    print(f"  {'-'*85}")
    print(f"  (* = best or within 0.001 of best)\n")

    # Win counts
    print("  Win counts:")
    for m in models_active:
        wins = sum(1 for _, r in df.iterrows()
                   if r[m] == max(r[c] for c in models_active))
        print(f"    {m}: {wins}/{len(df)}")

    # Average rank
    print("\n  Average rank (1=best):")
    ranks = df[models_active].rank(axis=1, ascending=False)
    for m in models_active:
        print(f"    {m}: {ranks[m].mean():.2f}")

    print(f"\n  Total wall-clock: {elapsed/60:.1f} min")

    return df


# ==========================================
# RUN
# ==========================================
df = run_benchmark()


Device: cuda
GPU: Tesla T4

Loading 8 datasets...

  California:  (20640, 8)
  Yacht:       (308, 6)
  Energy:      (768, 8)
  Concrete:    (1030, 8)
  Airfoil:     (1503, 5)
  Abalone:     (4177, 7)
  Kin8nm:      (8192, 8)
  Protein:     (45730, 9)

8 datasets ready.

Models: OLS, RF, Poly
Repeats: 5, Test: 0.2, Max N: 5000

  California (N=5000, P=8)
  Rep 1/5... done
  Rep 2/5... done
  Rep 3/5... done
  Rep 4/5... done
  Rep 5/5... done

  Model     R² (mean)    ± std   Time/rep
  ----------------------------------------
  OLS          0.5973   0.0292       0.0s
  RF           0.7765   0.0115       5.4s
  Poly         0.7610   0.0233      18.8s

  Yacht (N=308, P=6)
  Rep 1/5... done
  Rep 2/5... done
  Rep 3/5... done
  Rep 4/5... done
  Rep 5/5... done

  Model     R² (mean)    ± std   Time/rep
  ----------------------------------------
  OLS          0.5619   0.0880       0.0s
  RF           0.9787   0.0092       1.0s
  Poly         0.6865   0.0706       1.0s

  Energy (N=768, 

In [ ]:
"""
========================================================
REAL DATA BENCHMARK FOR ATTENTION REGRESSION — v4
With Polynomial Cross-Feature Transformer (Poly PCA+OLS)

Paste in a NEW Colab cell after running simulation cell.
Run cell 1 first (even briefly) to load function defs.
========================================================
"""

# ==========================================
# CONFIG — EDIT THIS SECTION
# ==========================================

DATASETS = {
    # --- Small & fast ---
    'Diabetes':    False,       # 442 obs, 10 feat (sklearn built-in)
    'Yacht':       True,       # 308 obs, 6 feat
    'Energy':      True,       # 768 obs, 8 feat
    # --- Medium ---
    'Concrete':    True,       # 1030 obs, 8 feat (nonlinear!)
    'Airfoil':     True,       # 1503 obs, 5 feat
    'WineRed':     False,       # 1599 obs, 11 feat
    # --- Larger ---
    'Abalone':     True,       # 4177 obs, 8 feat
    'California':  True,       # 20640→5000 obs, 8 feat (sklearn built-in)
    'Kin8nm':      True,       # 8192→5000 obs, 8 feat
    'Protein':     True,       # 45730→5000 obs, 9 feat
}

N_REPEATS = 5
TEST_FRAC = 0.2
MAX_N = 5000
SEED = 42

# Models
RUN_OLS  = True
RUN_RF   = True
RUN_GBM  = False
RUN_MLP  = False
RUN_ATTN = False
RUN_FTT  = True
RUN_POLY = True

# Hyperparameters
RF_TREES = 500;  RF_MTRY = 1/3
GBM_TREES = 500; GBM_LR = 0.01
MLP_LAYERS = 3;  MLP_UNITS = 200; MLP_DROP = 0.2
MLP_ENS = 1;    MLP_EPOCHS = 1000; MLP_PAT = 20
MLP_BS = 64;     MLP_LR_RATE = 1e-3; MLP_VAL = 0.15
ATTN_M = 5;      ATTN_STEPS = 300; ATTN_LR = 1.0; ATTN_REG = 1e-3

# FT-Transformer hyperparameters (Gorishniy et al. 2021)
FTT_D_MODEL = 64
FTT_N_HEADS = 4
FTT_FFN_DIM = 128
FTT_N_BLOCKS = 3
FTT_DROPOUT = 0.1
FTT_EPOCHS = 50
FTT_BS = 256
FTT_LR = 1e-3
FTT_WD = 1e-4

# Poly PCA+OLS hyperparameters
POLY_PCA_COMP = 200
POLY_D_MODEL = 64
POLY_FFN_DIM = 128
POLY_EPOCHS = 50
POLY_BS = 256
POLY_LR = 1e-3
POLY_DROPOUT = 0.1
POLY_WD = 1e-4

# ==========================================
# END CONFIG
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing, load_diabetes, fetch_openml
from sklearn.decomposition import PCA
import warnings, time
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ==========================================
# FT-TRANSFORMER (Gorishniy et al. 2021)
# ==========================================

def train_ft_transformer(X_train, y_train, X_test,
                         d_model=64, n_heads=4, ffn_dim=128,
                         n_blocks=3, dropout=0.1,
                         epochs=50, batch_size=256, lr=1e-3,
                         weight_decay=1e-4, seed=42, verbose=False):
    """
    FT-Transformer: Feature Tokenizer + [CLS] + Transformer Encoder.
    Per-feature learned embeddings, CLS token for readout.

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    class FTTransformer(nn.Module):
        def __init__(self, nf, d, nh, ffn, nblk, drop):
            super().__init__()
            self.feat_embeds = nn.ModuleList([nn.Linear(1, d) for _ in range(nf)])
            self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d, nhead=nh, dim_feedforward=ffn, dropout=drop,
                batch_first=True, activation='gelu')
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=nblk)
            self.norm = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)

        def forward(self, x):
            B = x.size(0)
            tokens = torch.stack([self.feat_embeds[i](x[:, i:i+1]) for i in range(x.size(1))], dim=1)
            tokens = torch.cat([self.cls_token.expand(B, -1, -1), tokens], dim=1)
            z = self.transformer(tokens)
            return self.head(self.norm(z[:, 0, :]))

    model = FTTransformer(nf, D, n_heads, ffn_dim, n_blocks, dropout).to(device)

    Xt = torch.from_numpy(X_train.astype(np.float32))
    yt = torch.from_numpy(y_train.astype(np.float32)).unsqueeze(1)
    tr_loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()

    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb); opt.zero_grad(); loss.backward(); opt.step()
        if verbose and (ep % 10 == 0 or ep == 1):
            model.eval()
            with torch.no_grad():
                r2 = 1 - np.mean((model(Xt.to(device)).cpu().numpy().ravel() - y_train)**2) / np.var(y_train)
            print(f"  FTT ep {ep}: train R²={r2:.4f}")

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters())}


# ==========================================
# POLY PCA+OLS MODEL
# ==========================================

def train_poly_model(X_train, y_train, X_test,
                     n_components=200, d_model=64, ffn_dim=128,
                     epochs=50, batch_size=256, lr=1e-3,
                     dropout=0.1, weight_decay=1e-4,
                     seed=42, verbose=False):
    """
    Polynomial Cross-Feature Regression Transformer with PCA + OLS init.
    Replaces self-attention with explicit polynomial interaction features.

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    def poly_cross(x):
        B, S, Dm = x.shape
        per = torch.cat([x, x**2 - 1], dim=-1)
        pairs = torch.cat([x[:,i,:]*x[:,j,:] for i in range(S) for j in range(i,S)], dim=-1)
        return torch.cat([per, pairs.unsqueeze(1).expand(B,S,-1)], dim=-1)

    class FrozenPCA(nn.Module):
        def __init__(self, mean, components):
            super().__init__()
            self.register_buffer('mean', torch.from_numpy(mean.astype(np.float32)))
            self.register_buffer('comp', torch.from_numpy(components.astype(np.float32)))
            self.out_dim = components.shape[0]
        def forward(self, x): return (x - self.mean) @ self.comp.T

    class PolyPCAModel(nn.Module):
        def __init__(self, nf, pca_layer, d, ffn_dim, drop):
            super().__init__()
            self.pca = pca_layer
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            self.W = nn.Linear(pca_layer.out_dim, d)
            self.n1 = nn.LayerNorm(d)
            self.ffn = nn.Sequential(nn.Linear(d, ffn_dim), nn.ReLU(), nn.Dropout(drop),
                                     nn.Linear(ffn_dim, d), nn.Dropout(drop))
            self.n2 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)
        def embed_x(self, x): return self.embed(x.unsqueeze(-1)) + self.pos
        def get_poly(self, emb): return self.pca(poly_cross(emb))
        def forward(self, x):
            emb = self.embed_x(x); phi = self.get_poly(emb)
            z = self.n1(emb + self.W(phi))
            return self.head(self.n2(z + self.ffn(z)).mean(1))

    # Step 1: PCA on training poly features
    torch.manual_seed(seed)
    tmp_embed = nn.Linear(1, D); tmp_pos = nn.Parameter(torch.randn(1, nf, D)*0.02)
    Xt = torch.from_numpy(X_train.astype(np.float32))
    with torch.no_grad():
        emb = tmp_embed(Xt.unsqueeze(-1)) + tmp_pos
        phi_all = poly_cross(emb).reshape(-1, emb.shape[-1]*2 + D*nf*(nf+1)//2)
    n_comp = min(n_components, phi_all.shape[1], phi_all.shape[0])
    pca = PCA(n_components=n_comp, random_state=seed); pca.fit(phi_all.numpy())
    pca_layer = FrozenPCA(pca.mean_, pca.components_)

    # Step 2: Build model
    torch.manual_seed(seed)
    model = PolyPCAModel(nf, pca_layer, D, ffn_dim, dropout).to(device)

    # Step 3: OLS warm-start
    model.eval()
    with torch.no_grad():
        phi_pooled = model.get_poly(model.embed_x(Xt.to(device))).mean(1).cpu().numpy()
    reg = Ridge(alpha=1.0); reg.fit(phi_pooled, y_train)
    with torch.no_grad():
        model.W.weight.data[0] = torch.from_numpy(reg.coef_.astype(np.float32)).to(device)
        model.W.bias.data.zero_()
        model.head.weight.data.zero_(); model.head.weight.data[0,0] = 1.0
        model.head.bias.data.fill_(float(reg.intercept_))

    # Step 4: Train
    yt = torch.from_numpy(y_train.astype(np.float32)).unsqueeze(1)
    tr_loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()
    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb); opt.zero_grad(); loss.backward(); opt.step()
        if verbose and (ep % 10 == 0 or ep == 1):
            model.eval()
            with torch.no_grad():
                r2 = 1 - np.mean((model(Xt.to(device)).cpu().numpy().ravel() - y_train)**2) / np.var(y_train)
            print(f"  Poly ep {ep}: train R²={r2:.4f}")

    # Step 5: Predict
    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters()), 'pca_components': n_comp}


# ==========================================
# DATASET LOADING — VALIDATED
# ==========================================

OPENML_REGISTRY = {
    'Concrete':  (44959,  (1000, 1100),  8,  'Concrete compressive strength'),
    'Energy':    (41478,  (700, 800),     8,  'Energy efficiency'),
    'Kin8nm':    (189,    (8000, 8300),   8,  'Kin8nm robot arm'),
    'Protein':   (44963,  (45000, 46000), 9,  'Protein structure CASP'),
    'WineRed':   (40691,  (1500, 1700),   11, 'Wine quality red'),
    'Yacht':     (42370,  (300, 320),     6,  'Yacht hydrodynamics'),
    'Airfoil':   (44957,  (1400, 1600),   5,  'Airfoil self-noise'),
    'Abalone':   (183,    (4100, 4200),   7,  'Abalone age prediction'),
}


def load_datasets():
    enabled = [k for k, v in DATASETS.items() if v]
    print(f"\n📦 Loading {len(enabled)} datasets: {', '.join(enabled)}\n")
    loaded = {}

    if 'California' in enabled:
        try:
            d = fetch_california_housing()
            assert d.data.shape == (20640, 8)
            loaded['California'] = (d.data, d.target)
            print(f"  ✓ California:  (20640, 8)")
        except Exception as e:
            print(f"  ✗ California: {e}")

    if 'Diabetes' in enabled:
        try:
            d = load_diabetes()
            assert d.data.shape == (442, 10)
            loaded['Diabetes'] = (d.data, d.target)
            print(f"  ✓ Diabetes:    (442, 10)")
        except Exception as e:
            print(f"  ✗ Diabetes: {e}")

    for name in enabled:
        if name in ('California', 'Diabetes'):
            continue
        if name not in OPENML_REGISTRY:
            print(f"  ⚠ {name}: not in registry, skipping")
            continue

        data_id, (n_min, n_max), expected_p, desc = OPENML_REGISTRY[name]
        try:
            data = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
            X = data.data.select_dtypes(include=[np.number]).values
            y_raw = data.target
            if y_raw is None:
                raise ValueError("Target column not set in OpenML metadata")
            if hasattr(y_raw, 'shape') and len(y_raw.shape) > 1 and y_raw.shape[1] > 1:
                y_raw = y_raw.iloc[:, 0]
            elif hasattr(y_raw, 'columns'):
                y_raw = y_raw.iloc[:, 0]
            if hasattr(y_raw, 'cat') and y_raw.dtype.name == 'category':
                y = y_raw.cat.codes.values.astype(float)
            else:
                y = np.array(y_raw).astype(float)
            mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
            X, y = X[mask], y[mask]
            N, P = X.shape
            if not (n_min <= N <= n_max):
                print(f"  ✗ {name} (id={data_id}): N={N} outside expected range ({n_min}-{n_max})")
                continue
            if P != expected_p:
                print(f"  ✗ {name} (id={data_id}): P={P} != expected {expected_p}")
                continue
            loaded[name] = (X, y)
            print(f"  ✓ {name}:{' '*(10-len(name))}({N}, {P})  — {desc}")
        except Exception as e:
            print(f"  ✗ {name} (id={data_id}): {e}")

    print(f"\n→ {len(loaded)}/{len(enabled)} datasets ready.\n")
    return loaded


# ==========================================
# BENCHMARK
# ==========================================

def run_benchmark():
    datasets = load_datasets()
    if not datasets:
        print("❌ No datasets loaded.")
        return None

    # Check function availability
    try:
        train_torch_model; has_attn = RUN_ATTN
    except NameError:
        has_attn = False
        if RUN_ATTN: print("⚠ train_torch_model not found. Run simulation cell first. Skipping Attn.\n")

    try:
        train_mlp_ensemble; has_mlp = RUN_MLP
    except NameError:
        has_mlp = False
        if RUN_MLP: print("⚠ train_mlp_ensemble not found. Run simulation cell first. Skipping MLP.\n")

    models_active = [m for m, on in [('OLS', RUN_OLS), ('RF', RUN_RF),
                     ('GBM', RUN_GBM), ('MLP', has_mlp), ('Attn', has_attn),
                     ('FTT', RUN_FTT), ('Poly', RUN_POLY)] if on]
    print(f"🔧 Models: {', '.join(models_active)}")
    print(f"🔁 Repeats: {N_REPEATS}, Test: {TEST_FRAC}, Max N: {MAX_N}\n")

    results = []
    t0 = time.time()

    for ds_name, (X_full, y_full) in datasets.items():
        N_full, P = X_full.shape

        if N_full > MAX_N:
            np.random.seed(SEED)
            idx = np.random.choice(N_full, MAX_N, replace=False)
            X_use, y_use = X_full[idx], y_full[idx]
            N_use = MAX_N
            label = f"{ds_name} (N={N_full}→{MAX_N}, P={P})"
        else:
            X_use, y_use = X_full, y_full
            N_use = N_full
            label = f"{ds_name} (N={N_use}, P={P})"

        print(f"{'='*70}\n  {label}\n{'='*70}")

        scores = {k: [] for k in models_active}
        times  = {k: [] for k in models_active}

        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            print(f"  Rep {r+1}/{N_REPEATS}...", end=" ", flush=True)

            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)

            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_train)
            X_te = scaler.transform(X_test)

            if 'OLS' in models_active:
                t_start = time.time()
                try:
                    m = LinearRegression().fit(X_tr, y_train)
                    scores['OLS'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['OLS'].append(np.nan)
                times['OLS'].append(time.time() - t_start)

            if 'RF' in models_active:
                t_start = time.time()
                try:
                    m = RandomForestRegressor(n_estimators=RF_TREES, max_features=RF_MTRY,
                                              n_jobs=-1, random_state=seed).fit(X_tr, y_train)
                    scores['RF'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['RF'].append(np.nan)
                times['RF'].append(time.time() - t_start)

            if 'GBM' in models_active:
                t_start = time.time()
                try:
                    m = GradientBoostingRegressor(n_estimators=GBM_TREES, learning_rate=GBM_LR,
                                                  random_state=seed).fit(X_tr, y_train)
                    scores['GBM'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['GBM'].append(np.nan)
                times['GBM'].append(time.time() - t_start)

            if 'MLP' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_mlp_ensemble(
                        X_tr, y_train, X_te, hidden_units=MLP_UNITS,
                        n_layers=MLP_LAYERS, dropout=MLP_DROP, ensemble_size=MLP_ENS,
                        max_epochs=MLP_EPOCHS, patience=MLP_PAT, batch_size=MLP_BS,
                        lr=MLP_LR_RATE, val_frac=MLP_VAL, seed=seed, verbose=False)
                    scores['MLP'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[MLP:{e}]", end=" ")
                    scores['MLP'].append(np.nan)
                times['MLP'].append(time.time() - t_start)

            if 'Attn' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_torch_model(
                        X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                        lr=ATTN_LR, reg_lambda=ATTN_REG, seed=seed, verbose=False)
                    r2_att = r2_score(y_test, y_pred)

                    if r2_att < 0.05:
                        y_pred, _ = train_torch_model(
                            X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                            lr=ATTN_LR, reg_lambda=ATTN_REG/100, seed=seed, verbose=False)
                        r2_att_retry = r2_score(y_test, y_pred)
                        if r2_att_retry > r2_att:
                            r2_att = r2_att_retry

                    if r2_att < 0:
                        y_var = np.var(y_test)
                        pred_var = np.var(y_pred)
                        pred_mean = np.mean(y_pred)
                        y_mean = np.mean(y_test)
                        mse = np.mean((y_pred - y_test)**2)
                        print(f"\n    ⚠ ATTN DIAGNOSTIC: R²={r2_att:.3f} | "
                              f"N_tr={len(y_train)} P={X_tr.shape[1]} | "
                              f"y_test: μ={y_mean:.2f} σ²={y_var:.2f} | "
                              f"ŷ: μ={pred_mean:.2f} σ²={pred_var:.2f} | "
                              f"MSE={mse:.2f}", end=" ")

                    scores['Attn'].append(r2_att)
                except Exception as e:
                    print(f"[Attn:{e}]", end=" ")
                    scores['Attn'].append(np.nan)
                times['Attn'].append(time.time() - t_start)

            if 'FTT' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_ft_transformer(
                        X_tr, y_train, X_te,
                        d_model=FTT_D_MODEL, n_heads=FTT_N_HEADS,
                        ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                        dropout=FTT_DROPOUT, epochs=FTT_EPOCHS,
                        batch_size=FTT_BS, lr=FTT_LR,
                        weight_decay=FTT_WD,
                        seed=seed, verbose=False)
                    scores['FTT'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[FTT:{e}]", end=" ")
                    scores['FTT'].append(np.nan)
                times['FTT'].append(time.time() - t_start)

            if 'Poly' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_poly_model(
                        X_tr, y_train, X_te,
                        n_components=POLY_PCA_COMP, d_model=POLY_D_MODEL,
                        ffn_dim=POLY_FFN_DIM, epochs=POLY_EPOCHS,
                        batch_size=POLY_BS, lr=POLY_LR,
                        dropout=POLY_DROPOUT, weight_decay=POLY_WD,
                        seed=seed, verbose=False)
                    scores['Poly'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[Poly:{e}]", end=" ")
                    scores['Poly'].append(np.nan)
                times['Poly'].append(time.time() - t_start)

            print("✓")

        avg = {k: np.nanmean(v) if v else np.nan for k, v in scores.items()}
        se  = {k: np.nanstd(v)  if v else np.nan for k, v in scores.items()}
        avg_t = {k: np.mean(v) if v else np.nan for k, v in times.items()}

        results.append({
            'Dataset': ds_name, 'N': N_use, 'P': P,
            **{m: avg[m] for m in models_active},
            **{f'{m}_se': se[m] for m in models_active},
            **{f'{m}_time': avg_t[m] for m in models_active},
        })

        parts = [f"{m}:{avg[m]:.3f}" for m in models_active if not np.isnan(avg[m])]
        valid = {k: v for k, v in avg.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '?'
        print(f"  → {' | '.join(parts)}  |  Winner: {best}\n")

    # ==========================================
    # OUTPUT
    # ==========================================
    elapsed = time.time() - t0
    df = pd.DataFrame(results)

    print("\n" + "="*120)
    print(f"RESULTS ({elapsed/60:.1f} min, {N_REPEATS} reps, device={DEVICE})")
    print("="*120)

    header = f"{'Dataset':<12} {'N':>5} {'P':>3}"
    for m in models_active: header += f"  {m:>7}"
    header += "  |"
    for m in models_active: header += f"  {'t_'+m:>7}"
    header += "   Best"
    print(header)
    print("-"*120)

    for _, row in df.iterrows():
        line = f"{row['Dataset']:<12} {int(row['N']):>5} {int(row['P']):>3}"
        vals = {}
        for m in models_active:
            v = row[m]
            vals[m] = v
            line += f"  {v:>7.4f}" if not np.isnan(v) else f"  {'--':>7}"
        line += "  |"
        for m in models_active:
            t = row.get(f'{m}_time', np.nan)
            line += f"  {t:>6.1f}s" if not np.isnan(t) else f"  {'--':>7}"
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '--'
        line += f"   {best}"
        print(line)

    print("-"*120)

    # Win counts
    print("\nWin counts:")
    for m in models_active:
        wins = sum(1 for _, r in df.iterrows()
                   if not np.isnan(r[m]) and r[m] == max(r[c] for c in models_active if not np.isnan(r[c])))
        print(f"  {m}: {wins}/{len(df)}")

    # Average rank
    print("\nAverage rank (1=best):")
    ranks = df[models_active].rank(axis=1, ascending=False)
    for m in models_active:
        print(f"  {m}: {ranks[m].mean():.2f}")

    # Timing summary
    print("\nAvg time per rep (seconds):")
    for m in models_active:
        avg_t = df[f'{m}_time'].mean()
        print(f"  {m}: {avg_t:.1f}s")

    df.to_csv("real_data_results_v4.csv", index=False)
    print(f"\n💾 Saved: real_data_results_v4.csv")

    # LaTeX
    col_names = {'OLS': 'OLS', 'RF': 'RF', 'GBM': 'GBM', 'MLP': 'MLP',
                 'Attn': r'Att.\ Reg.', 'FTT': 'FT-T', 'Poly': 'Poly'}
    print("\n\n% ========== LaTeX Table ==========")
    print(r"\begin{table}[htbp]")
    print(r"\centering")
    print(rf"\caption{{Out-of-sample $R^2$ on standard regression benchmarks ({N_REPEATS} random 80/20 splits, averaged). Best in bold.}}")
    print(r"\label{tab:real_data}")
    print(r"\begin{tabular}{lrr" + "c"*len(models_active) + "}")
    print(r"\toprule")
    hdr = r"Dataset & $N$ & $P$"
    for m in models_active: hdr += f" & {col_names.get(m, m)}"
    print(hdr + r" \\")
    print(r"\midrule")

    for _, row in df.iterrows():
        vals = {m: row[m] for m in models_active}
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best_val = max(valid.values()) if valid else None
        cells = []
        for m in models_active:
            v = row[m]
            if np.isnan(v): cells.append("--")
            elif best_val and abs(v - best_val) < 1e-4: cells.append(rf"\textbf{{{v:.3f}}}")
            else: cells.append(f"{v:.3f}")
        print(f"{row['Dataset']} & {int(row['N'])} & {int(row['P'])} & {' & '.join(cells)}" + r" \\")

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")

    return df


# ==========================================
# RUN
# ==========================================
df_results = run_benchmark()


Device: cuda
GPU: Tesla T4

📦 Loading 8 datasets: Yacht, Energy, Concrete, Airfoil, Abalone, California, Kin8nm, Protein

  ✓ California:  (20640, 8)
  ✓ Yacht:     (308, 6)  — Yacht hydrodynamics
  ✓ Energy:    (768, 8)  — Energy efficiency
  ✓ Concrete:  (1030, 8)  — Concrete compressive strength
  ✓ Airfoil:   (1503, 5)  — Airfoil self-noise
  ✓ Abalone:   (4177, 7)  — Abalone age prediction
  ✓ Kin8nm:    (8192, 8)  — Kin8nm robot arm
  ✓ Protein:   (45730, 9)  — Protein structure CASP

→ 8/8 datasets ready.

🔧 Models: OLS, RF, FTT, Poly
🔁 Repeats: 5, Test: 0.2, Max N: 5000

  California (N=20640→5000, P=8)
  Rep 1/5... ✓
  Rep 2/5... ✓
  Rep 3/5... ✓
  Rep 4/5... ✓
  Rep 5/5... ✓
  → OLS:0.597 | RF:0.777 | FTT:0.777 | Poly:0.761  |  Winner: FTT

  Yacht (N=308, P=6)
  Rep 1/5... ✓
  Rep 2/5... ✓
  Rep 3/5... ✓
  Rep 4/5... ✓
  Rep 5/5... ✓
  → OLS:0.562 | RF:0.979 | FTT:0.167 | Poly:0.687  |  Winner: RF

  Energy (N=768, P=8)
  Rep 1/5... ✓
  Rep 2/5... ✓
  Rep 3/5... ✓
  Rep 4/5.

In [ ]:
"""
========================================================
REAL DATA BENCHMARK FOR ATTENTION REGRESSION — v4
With Polynomial Cross-Feature Transformer (Poly PCA+OLS)

Paste in a NEW Colab cell after running simulation cell.
Run cell 1 first (even briefly) to load function defs.
========================================================
"""

# ==========================================
# CONFIG — EDIT THIS SECTION
# ==========================================

DATASETS = {
    # --- Small & fast ---
    'Diabetes':    False,       # 442 obs, 10 feat (sklearn built-in)
    'Yacht':       True,       # 308 obs, 6 feat
    'Energy':      True,       # 768 obs, 8 feat
    # --- Medium ---
    'Concrete':    True,       # 1030 obs, 8 feat (nonlinear!)
    'Airfoil':     True,       # 1503 obs, 5 feat
    'WineRed':     False,       # 1599 obs, 11 feat
    # --- Larger ---
    'Abalone':     True,       # 4177 obs, 8 feat
    'California':  True,       # 20640→5000 obs, 8 feat (sklearn built-in)
    'Kin8nm':      True,       # 8192→5000 obs, 8 feat
    'Protein':     True,       # 45730→5000 obs, 9 feat
}

N_REPEATS = 2
TEST_FRAC = 0.2
MAX_N = 5000
SEED = 42

# Models
RUN_OLS  = True
RUN_RF   = True
RUN_GBM  = False
RUN_MLP  = False
RUN_ATTN = True
RUN_FTT  = True
RUN_POLY = True

# Hyperparameters
RF_TREES = 500;  RF_MTRY = 1/3
GBM_TREES = 500; GBM_LR = 0.01
MLP_LAYERS = 3;  MLP_UNITS = 200; MLP_DROP = 0.2
MLP_ENS = 1;    MLP_EPOCHS = 1000; MLP_PAT = 20
MLP_BS = 64;     MLP_LR_RATE = 1e-3; MLP_VAL = 0.15
ATTN_M = 5;      ATTN_STEPS = 300; ATTN_LR = 1.0; ATTN_REG = 1e-3

# FT-Transformer hyperparameters (Gorishniy et al. 2021)
FTT_D_MODEL = 64
FTT_N_HEADS = 4
FTT_FFN_DIM = 128
FTT_N_BLOCKS = 3
FTT_DROPOUT = 0.1
FTT_EPOCHS = 100
FTT_BS = 256
FTT_LR = 1e-3
FTT_WD = 1e-4

# Poly PCA+OLS hyperparameters
POLY_PCA_COMP = 200
POLY_D_MODEL = 64
POLY_FFN_DIM = 128
POLY_EPOCHS = 100
POLY_BS = 256
POLY_LR = 1e-3
POLY_DROPOUT = 0.1
POLY_WD = 1e-4

# ==========================================
# END CONFIG
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing, load_diabetes, fetch_openml
from sklearn.decomposition import PCA
import warnings, time
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ==========================================
# FT-TRANSFORMER (Gorishniy et al. 2021)
# ==========================================

def train_ft_transformer(X_train, y_train, X_test,
                         d_model=64, n_heads=4, ffn_dim=128,
                         n_blocks=3, dropout=0.1,
                         epochs=50, batch_size=256, lr=1e-3,
                         weight_decay=1e-4, seed=42, verbose=False):
    """
    FT-Transformer: Feature Tokenizer + [CLS] + Transformer Encoder.
    Per-feature learned embeddings, CLS token for readout.

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    class FTTransformer(nn.Module):
        def __init__(self, nf, d, nh, ffn, nblk, drop):
            super().__init__()
            self.feat_embeds = nn.ModuleList([nn.Linear(1, d) for _ in range(nf)])
            self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d, nhead=nh, dim_feedforward=ffn, dropout=drop,
                batch_first=True, activation='gelu')
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=nblk)
            self.norm = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)

        def forward(self, x):
            B = x.size(0)
            tokens = torch.stack([self.feat_embeds[i](x[:, i:i+1]) for i in range(x.size(1))], dim=1)
            tokens = torch.cat([self.cls_token.expand(B, -1, -1), tokens], dim=1)
            z = self.transformer(tokens)
            return self.head(self.norm(z[:, 0, :]))

    model = FTTransformer(nf, D, n_heads, ffn_dim, n_blocks, dropout).to(device)

    # Val split for early stopping (15%)
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    crit = nn.MSELoss()

    # LR warmup + cosine decay
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    warmup_epochs = max(1, epochs // 10)
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return ep / warmup_epochs
        progress = (ep - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    # Training with early stopping + grad clipping
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10

    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()

        # Val check
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xt_val), yt_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Restore best
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters())}


# ==========================================
# POLY PCA+OLS MODEL
# ==========================================

def train_poly_model(X_train, y_train, X_test,
                     n_components=200, d_model=64, ffn_dim=128,
                     epochs=50, batch_size=256, lr=1e-3,
                     dropout=0.1, weight_decay=1e-4,
                     seed=42, verbose=False):
    """
    Polynomial Cross-Feature Regression Transformer with PCA + OLS init.
    Replaces self-attention with explicit polynomial interaction features.

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    def poly_cross(x):
        B, S, Dm = x.shape
        per = torch.cat([x, x**2 - 1], dim=-1)
        pairs = torch.cat([x[:,i,:]*x[:,j,:] for i in range(S) for j in range(i,S)], dim=-1)
        return torch.cat([per, pairs.unsqueeze(1).expand(B,S,-1)], dim=-1)

    class FrozenPCA(nn.Module):
        def __init__(self, mean, components):
            super().__init__()
            self.register_buffer('mean', torch.from_numpy(mean.astype(np.float32)))
            self.register_buffer('comp', torch.from_numpy(components.astype(np.float32)))
            self.out_dim = components.shape[0]
        def forward(self, x): return (x - self.mean) @ self.comp.T

    class PolyPCAModel(nn.Module):
        def __init__(self, nf, pca_layer, d, ffn_dim, drop):
            super().__init__()
            self.pca = pca_layer
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            self.W = nn.Linear(pca_layer.out_dim, d)
            self.n1 = nn.LayerNorm(d)
            self.ffn = nn.Sequential(nn.Linear(d, ffn_dim), nn.ReLU(), nn.Dropout(drop),
                                     nn.Linear(ffn_dim, d), nn.Dropout(drop))
            self.n2 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)
        def embed_x(self, x): return self.embed(x.unsqueeze(-1)) + self.pos
        def get_poly(self, emb): return self.pca(poly_cross(emb))
        def forward(self, x):
            emb = self.embed_x(x); phi = self.get_poly(emb)
            z = self.n1(emb + self.W(phi))
            return self.head(self.n2(z + self.ffn(z)).mean(1))

    # Step 1: PCA on training poly features
    torch.manual_seed(seed)
    tmp_embed = nn.Linear(1, D); tmp_pos = nn.Parameter(torch.randn(1, nf, D)*0.02)
    Xt = torch.from_numpy(X_train.astype(np.float32))
    with torch.no_grad():
        emb = tmp_embed(Xt.unsqueeze(-1)) + tmp_pos
        phi_all = poly_cross(emb).reshape(-1, emb.shape[-1]*2 + D*nf*(nf+1)//2)
    n_comp = min(n_components, phi_all.shape[1], phi_all.shape[0])
    pca = PCA(n_components=n_comp, random_state=seed); pca.fit(phi_all.numpy())
    pca_layer = FrozenPCA(pca.mean_, pca.components_)

    # Step 2: Build model
    torch.manual_seed(seed)
    model = PolyPCAModel(nf, pca_layer, D, ffn_dim, dropout).to(device)

    # Step 3: OLS warm-start
    model.eval()
    with torch.no_grad():
        phi_pooled = model.get_poly(model.embed_x(Xt.to(device))).mean(1).cpu().numpy()
    reg = Ridge(alpha=1.0); reg.fit(phi_pooled, y_train)
    with torch.no_grad():
        model.W.weight.data[0] = torch.from_numpy(reg.coef_.astype(np.float32)).to(device)
        model.W.bias.data.zero_()
        model.head.weight.data.zero_(); model.head.weight.data[0,0] = 1.0
        model.head.bias.data.fill_(float(reg.intercept_))

    # Step 4: Train with early stopping + grad clipping (same as FTT for fairness)
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10

    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xt_val), yt_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # Step 5: Predict
    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters()), 'pca_components': n_comp}


# ==========================================
# DATASET LOADING — VALIDATED
# ==========================================

OPENML_REGISTRY = {
    'Concrete':  (44959,  (1000, 1100),  8,  'Concrete compressive strength'),
    'Energy':    (41478,  (700, 800),     8,  'Energy efficiency'),
    'Kin8nm':    (189,    (8000, 8300),   8,  'Kin8nm robot arm'),
    'Protein':   (44963,  (45000, 46000), 9,  'Protein structure CASP'),
    'WineRed':   (40691,  (1500, 1700),   11, 'Wine quality red'),
    'Yacht':     (42370,  (300, 320),     6,  'Yacht hydrodynamics'),
    'Airfoil':   (44957,  (1400, 1600),   5,  'Airfoil self-noise'),
    'Abalone':   (183,    (4100, 4200),   7,  'Abalone age prediction'),
}


def load_datasets():
    enabled = [k for k, v in DATASETS.items() if v]
    print(f"\n📦 Loading {len(enabled)} datasets: {', '.join(enabled)}\n")
    loaded = {}

    if 'California' in enabled:
        try:
            d = fetch_california_housing()
            assert d.data.shape == (20640, 8)
            loaded['California'] = (d.data, d.target)
            print(f"  ✓ California:  (20640, 8)")
        except Exception as e:
            print(f"  ✗ California: {e}")

    if 'Diabetes' in enabled:
        try:
            d = load_diabetes()
            assert d.data.shape == (442, 10)
            loaded['Diabetes'] = (d.data, d.target)
            print(f"  ✓ Diabetes:    (442, 10)")
        except Exception as e:
            print(f"  ✗ Diabetes: {e}")

    for name in enabled:
        if name in ('California', 'Diabetes'):
            continue
        if name not in OPENML_REGISTRY:
            print(f"  ⚠ {name}: not in registry, skipping")
            continue

        data_id, (n_min, n_max), expected_p, desc = OPENML_REGISTRY[name]
        try:
            data = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
            X = data.data.select_dtypes(include=[np.number]).values
            y_raw = data.target
            if y_raw is None:
                raise ValueError("Target column not set in OpenML metadata")
            if hasattr(y_raw, 'shape') and len(y_raw.shape) > 1 and y_raw.shape[1] > 1:
                y_raw = y_raw.iloc[:, 0]
            elif hasattr(y_raw, 'columns'):
                y_raw = y_raw.iloc[:, 0]
            if hasattr(y_raw, 'cat') and y_raw.dtype.name == 'category':
                y = y_raw.cat.codes.values.astype(float)
            else:
                y = np.array(y_raw).astype(float)
            mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
            X, y = X[mask], y[mask]
            N, P = X.shape
            if not (n_min <= N <= n_max):
                print(f"  ✗ {name} (id={data_id}): N={N} outside expected range ({n_min}-{n_max})")
                continue
            if P != expected_p:
                print(f"  ✗ {name} (id={data_id}): P={P} != expected {expected_p}")
                continue
            loaded[name] = (X, y)
            print(f"  ✓ {name}:{' '*(10-len(name))}({N}, {P})  — {desc}")
        except Exception as e:
            print(f"  ✗ {name} (id={data_id}): {e}")

    print(f"\n→ {len(loaded)}/{len(enabled)} datasets ready.\n")
    return loaded


# ==========================================
# BENCHMARK
# ==========================================

def run_benchmark():
    datasets = load_datasets()
    if not datasets:
        print("❌ No datasets loaded.")
        return None

    # Check function availability
    try:
        train_torch_model; has_attn = RUN_ATTN
    except NameError:
        has_attn = False
        if RUN_ATTN: print("⚠ train_torch_model not found. Run simulation cell first. Skipping Attn.\n")

    try:
        train_mlp_ensemble; has_mlp = RUN_MLP
    except NameError:
        has_mlp = False
        if RUN_MLP: print("⚠ train_mlp_ensemble not found. Run simulation cell first. Skipping MLP.\n")

    models_active = [m for m, on in [('OLS', RUN_OLS), ('RF', RUN_RF),
                     ('GBM', RUN_GBM), ('MLP', has_mlp), ('Attn', has_attn),
                     ('FTT', RUN_FTT), ('Poly', RUN_POLY)] if on]
    print(f"🔧 Models: {', '.join(models_active)}")
    print(f"🔁 Repeats: {N_REPEATS}, Test: {TEST_FRAC}, Max N: {MAX_N}\n")

    results = []
    t0 = time.time()

    for ds_name, (X_full, y_full) in datasets.items():
        N_full, P = X_full.shape

        if N_full > MAX_N:
            np.random.seed(SEED)
            idx = np.random.choice(N_full, MAX_N, replace=False)
            X_use, y_use = X_full[idx], y_full[idx]
            N_use = MAX_N
            label = f"{ds_name} (N={N_full}→{MAX_N}, P={P})"
        else:
            X_use, y_use = X_full, y_full
            N_use = N_full
            label = f"{ds_name} (N={N_use}, P={P})"

        print(f"{'='*70}\n  {label}\n{'='*70}")

        scores = {k: [] for k in models_active}
        times  = {k: [] for k in models_active}

        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            print(f"  Rep {r+1}/{N_REPEATS}...", end=" ", flush=True)

            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)

            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_train)
            X_te = scaler.transform(X_test)

            if 'OLS' in models_active:
                t_start = time.time()
                try:
                    m = LinearRegression().fit(X_tr, y_train)
                    scores['OLS'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['OLS'].append(np.nan)
                times['OLS'].append(time.time() - t_start)

            if 'RF' in models_active:
                t_start = time.time()
                try:
                    m = RandomForestRegressor(n_estimators=RF_TREES, max_features=RF_MTRY,
                                              n_jobs=-1, random_state=seed).fit(X_tr, y_train)
                    scores['RF'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['RF'].append(np.nan)
                times['RF'].append(time.time() - t_start)

            if 'GBM' in models_active:
                t_start = time.time()
                try:
                    m = GradientBoostingRegressor(n_estimators=GBM_TREES, learning_rate=GBM_LR,
                                                  random_state=seed).fit(X_tr, y_train)
                    scores['GBM'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['GBM'].append(np.nan)
                times['GBM'].append(time.time() - t_start)

            if 'MLP' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_mlp_ensemble(
                        X_tr, y_train, X_te, hidden_units=MLP_UNITS,
                        n_layers=MLP_LAYERS, dropout=MLP_DROP, ensemble_size=MLP_ENS,
                        max_epochs=MLP_EPOCHS, patience=MLP_PAT, batch_size=MLP_BS,
                        lr=MLP_LR_RATE, val_frac=MLP_VAL, seed=seed, verbose=False)
                    scores['MLP'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[MLP:{e}]", end=" ")
                    scores['MLP'].append(np.nan)
                times['MLP'].append(time.time() - t_start)

            if 'Attn' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_torch_model(
                        X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                        lr=ATTN_LR, reg_lambda=ATTN_REG, seed=seed, verbose=False)
                    r2_att = r2_score(y_test, y_pred)

                    if r2_att < 0.05:
                        y_pred, _ = train_torch_model(
                            X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                            lr=ATTN_LR, reg_lambda=ATTN_REG/100, seed=seed, verbose=False)
                        r2_att_retry = r2_score(y_test, y_pred)
                        if r2_att_retry > r2_att:
                            r2_att = r2_att_retry

                    if r2_att < 0:
                        y_var = np.var(y_test)
                        pred_var = np.var(y_pred)
                        pred_mean = np.mean(y_pred)
                        y_mean = np.mean(y_test)
                        mse = np.mean((y_pred - y_test)**2)
                        print(f"\n    ⚠ ATTN DIAGNOSTIC: R²={r2_att:.3f} | "
                              f"N_tr={len(y_train)} P={X_tr.shape[1]} | "
                              f"y_test: μ={y_mean:.2f} σ²={y_var:.2f} | "
                              f"ŷ: μ={pred_mean:.2f} σ²={pred_var:.2f} | "
                              f"MSE={mse:.2f}", end=" ")

                    scores['Attn'].append(r2_att)
                except Exception as e:
                    print(f"[Attn:{e}]", end=" ")
                    scores['Attn'].append(np.nan)
                times['Attn'].append(time.time() - t_start)

            if 'FTT' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_ft_transformer(
                        X_tr, y_train, X_te,
                        d_model=FTT_D_MODEL, n_heads=FTT_N_HEADS,
                        ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                        dropout=FTT_DROPOUT, epochs=FTT_EPOCHS,
                        batch_size=FTT_BS, lr=FTT_LR,
                        weight_decay=FTT_WD,
                        seed=seed, verbose=False)
                    scores['FTT'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[FTT:{e}]", end=" ")
                    scores['FTT'].append(np.nan)
                times['FTT'].append(time.time() - t_start)

            if 'Poly' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_poly_model(
                        X_tr, y_train, X_te,
                        n_components=POLY_PCA_COMP, d_model=POLY_D_MODEL,
                        ffn_dim=POLY_FFN_DIM, epochs=POLY_EPOCHS,
                        batch_size=POLY_BS, lr=POLY_LR,
                        dropout=POLY_DROPOUT, weight_decay=POLY_WD,
                        seed=seed, verbose=False)
                    scores['Poly'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[Poly:{e}]", end=" ")
                    scores['Poly'].append(np.nan)
                times['Poly'].append(time.time() - t_start)

            print("✓")

        avg = {k: np.nanmean(v) if v else np.nan for k, v in scores.items()}
        se  = {k: np.nanstd(v)  if v else np.nan for k, v in scores.items()}
        avg_t = {k: np.mean(v) if v else np.nan for k, v in times.items()}

        results.append({
            'Dataset': ds_name, 'N': N_use, 'P': P,
            **{m: avg[m] for m in models_active},
            **{f'{m}_se': se[m] for m in models_active},
            **{f'{m}_time': avg_t[m] for m in models_active},
        })

        parts = [f"{m}:{avg[m]:.3f}" for m in models_active if not np.isnan(avg[m])]
        valid = {k: v for k, v in avg.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '?'
        print(f"  → {' | '.join(parts)}  |  Winner: {best}\n")

    # ==========================================
    # OUTPUT
    # ==========================================
    elapsed = time.time() - t0
    df = pd.DataFrame(results)

    print("\n" + "="*120)
    print(f"RESULTS ({elapsed/60:.1f} min, {N_REPEATS} reps, device={DEVICE})")
    print("="*120)

    header = f"{'Dataset':<12} {'N':>5} {'P':>3}"
    for m in models_active: header += f"  {m:>7}"
    header += "  |"
    for m in models_active: header += f"  {'t_'+m:>7}"
    header += "   Best"
    print(header)
    print("-"*120)

    for _, row in df.iterrows():
        line = f"{row['Dataset']:<12} {int(row['N']):>5} {int(row['P']):>3}"
        vals = {}
        for m in models_active:
            v = row[m]
            vals[m] = v
            line += f"  {v:>7.4f}" if not np.isnan(v) else f"  {'--':>7}"
        line += "  |"
        for m in models_active:
            t = row.get(f'{m}_time', np.nan)
            line += f"  {t:>6.1f}s" if not np.isnan(t) else f"  {'--':>7}"
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '--'
        line += f"   {best}"
        print(line)

    print("-"*120)

    # Win counts
    print("\nWin counts:")
    for m in models_active:
        wins = sum(1 for _, r in df.iterrows()
                   if not np.isnan(r[m]) and r[m] == max(r[c] for c in models_active if not np.isnan(r[c])))
        print(f"  {m}: {wins}/{len(df)}")

    # Average rank
    print("\nAverage rank (1=best):")
    ranks = df[models_active].rank(axis=1, ascending=False)
    for m in models_active:
        print(f"  {m}: {ranks[m].mean():.2f}")

    # Timing summary
    print("\nAvg time per rep (seconds):")
    for m in models_active:
        avg_t = df[f'{m}_time'].mean()
        print(f"  {m}: {avg_t:.1f}s")

    df.to_csv("real_data_results_v4.csv", index=False)
    print(f"\n💾 Saved: real_data_results_v4.csv")

    # LaTeX
    col_names = {'OLS': 'OLS', 'RF': 'RF', 'GBM': 'GBM', 'MLP': 'MLP',
                 'Attn': r'Att.\ Reg.', 'FTT': 'FT-T', 'Poly': 'Poly'}
    print("\n\n% ========== LaTeX Table ==========")
    print(r"\begin{table}[htbp]")
    print(r"\centering")
    print(rf"\caption{{Out-of-sample $R^2$ on standard regression benchmarks ({N_REPEATS} random 80/20 splits, averaged). Best in bold.}}")
    print(r"\label{tab:real_data}")
    print(r"\begin{tabular}{lrr" + "c"*len(models_active) + "}")
    print(r"\toprule")
    hdr = r"Dataset & $N$ & $P$"
    for m in models_active: hdr += f" & {col_names.get(m, m)}"
    print(hdr + r" \\")
    print(r"\midrule")

    for _, row in df.iterrows():
        vals = {m: row[m] for m in models_active}
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best_val = max(valid.values()) if valid else None
        cells = []
        for m in models_active:
            v = row[m]
            if np.isnan(v): cells.append("--")
            elif best_val and abs(v - best_val) < 1e-4: cells.append(rf"\textbf{{{v:.3f}}}")
            else: cells.append(f"{v:.3f}")
        print(f"{row['Dataset']} & {int(row['N'])} & {int(row['P'])} & {' & '.join(cells)}" + r" \\")

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")

    return df


# ==========================================
# RUN
# ==========================================
df_results = run_benchmark()


Device: cuda
GPU: Tesla T4

📦 Loading 8 datasets: Yacht, Energy, Concrete, Airfoil, Abalone, California, Kin8nm, Protein

  ✓ California:  (20640, 8)
  ✓ Yacht:     (308, 6)  — Yacht hydrodynamics
  ✓ Energy:    (768, 8)  — Energy efficiency
  ✓ Concrete:  (1030, 8)  — Concrete compressive strength
  ✓ Airfoil:   (1503, 5)  — Airfoil self-noise
  ✓ Abalone:   (4177, 7)  — Abalone age prediction
  ✓ Kin8nm:    (8192, 8)  — Kin8nm robot arm
  ✓ Protein:   (45730, 9)  — Protein structure CASP

→ 8/8 datasets ready.

⚠ train_torch_model not found. Run simulation cell first. Skipping Attn.

🔧 Models: OLS, RF, FTT, Poly
🔁 Repeats: 2, Test: 0.2, Max N: 5000

  California (N=20640→5000, P=8)
  Rep 1/2... ✓
  Rep 2/2... ✓
  → OLS:0.604 | RF:0.783 | FTT:0.770 | Poly:0.772  |  Winner: RF

  Yacht (N=308, P=6)
  Rep 1/2... ✓
  Rep 2/2... ✓
  → OLS:0.613 | RF:0.972 | FTT:0.122 | Poly:0.722  |  Winner: RF

  Energy (N=768, P=8)
  Rep 1/2... ✓
  Rep 2/2... ✓
  → OLS:0.909 | RF:0.996 | FTT:0.138 | Pol

In [ ]:
"""
========================================================
REAL DATA BENCHMARK FOR ATTENTION REGRESSION ‚Äî v4
With Polynomial Cross-Feature Transformer (Poly PCA+OLS)

Paste in a NEW Colab cell after running simulation cell.
Run cell 1 first (even briefly) to load function defs.
========================================================
"""

# ==========================================
# CONFIG ‚Äî EDIT THIS SECTION
# ==========================================

DATASETS = {
    # --- Small & fast ---
    'Diabetes':    False,       # 442 obs, 10 feat (sklearn built-in)
    'Yacht':       True,       # 308 obs, 6 feat
    'Energy':      True,       # 768 obs, 8 feat
    # --- Medium ---
    'Concrete':    True,       # 1030 obs, 8 feat (nonlinear!)
    'Airfoil':     True,       # 1503 obs, 5 feat
    'WineRed':     False,       # 1599 obs, 11 feat
    # --- Larger ---
    'Abalone':     True,       # 4177 obs, 8 feat
    'California':  True,       # 20640‚Üí5000 obs, 8 feat (sklearn built-in)
    'Kin8nm':      True,       # 8192‚Üí5000 obs, 8 feat
    'Protein':     True,       # 45730‚Üí5000 obs, 9 feat
}

N_REPEATS = 5
TEST_FRAC = 0.2
MAX_N = 5000
SEED = 42

# Models
RUN_OLS  = True
RUN_RF   = True
RUN_GBM  = False
RUN_MLP  = False
RUN_ATTN = True
RUN_FTT  = True
RUN_POLY = True

# Hyperparameters
RF_TREES = 500;  RF_MTRY = 1/3
GBM_TREES = 500; GBM_LR = 0.01
MLP_LAYERS = 3;  MLP_UNITS = 200; MLP_DROP = 0.2
MLP_ENS = 1;    MLP_EPOCHS = 1000; MLP_PAT = 20
MLP_BS = 64;     MLP_LR_RATE = 1e-3; MLP_VAL = 0.15
ATTN_M = 4;      ATTN_STEPS = 300; ATTN_LR = 1e-3; ATTN_REG = 1e-3

# FT-Transformer hyperparameters (Gorishniy et al. 2021)
FTT_D_MODEL = 64
FTT_N_HEADS = 4
FTT_FFN_DIM = 128
FTT_N_BLOCKS = 3
FTT_DROPOUT = 0.1
FTT_EPOCHS = 200
FTT_BS = 256
FTT_LR = 1e-3
FTT_WD = 1e-4

# Poly PCA+OLS hyperparameters
POLY_PCA_COMP = 200
POLY_D_MODEL = 64
POLY_FFN_DIM = 128
POLY_EPOCHS = 200
POLY_BS = 256
POLY_LR = 1e-3
POLY_DROPOUT = 0.1
POLY_WD = 1e-4

# ==========================================
# END CONFIG
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing, load_diabetes, fetch_openml
from sklearn.decomposition import PCA
import warnings, time
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ==========================================
# ATTENTION REGRESSION (simple 2-block)
# Self-contained ‚Äî no dependency on cell 1
# ==========================================

def train_torch_model(X_train, y_train, X_test,
                      M=4, steps=300, lr=1e-3,
                      reg_lambda=1e-3, seed=42, verbose=False):
    """
    Simple attention-based regression: embed features as tokens,
    2-block multi-head self-attention, pool, linear head.
    Trains for `steps` gradient steps (not epochs).
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = 64

    class AttnRegressor(nn.Module):
        def __init__(self, nf, d, nh):
            super().__init__()
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            self.attn1 = nn.MultiheadAttention(d, nh, batch_first=True, dropout=0.1)
            self.n1 = nn.LayerNorm(d)
            self.ff1 = nn.Sequential(nn.Linear(d, d*2), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d*2, d))
            self.n2 = nn.LayerNorm(d)
            self.attn2 = nn.MultiheadAttention(d, nh, batch_first=True, dropout=0.1)
            self.n3 = nn.LayerNorm(d)
            self.ff2 = nn.Sequential(nn.Linear(d, d*2), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d*2, d))
            self.n4 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)

        def forward(self, x):
            x = self.embed(x.unsqueeze(-1)) + self.pos
            a, _ = self.attn1(x, x, x); x = self.n1(x + a); x = self.n2(x + self.ff1(x))
            a, _ = self.attn2(x, x, x); x = self.n3(x + a); x = self.n4(x + self.ff2(x))
            return self.head(x.mean(1))

    model = AttnRegressor(nf, D, max(1, M)).to(device)

    # Val split for early stopping
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=256, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=reg_lambda)
    crit = nn.MSELoss()

    best_val = float('inf'); best_state = None; patience = 0; step = 0
    done = False
    while not done:
        model.train()
        for Xb, yb in tr_loader:
            if step >= steps:
                done = True; break
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); step += 1

        # Val check each pass through data
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xt_val), yt_val).item()
        if vl < best_val:
            best_val = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= 10: break

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters())}


def train_mlp_ensemble(*args, **kwargs):
    """Stub ‚Äî not implemented in standalone mode."""
    raise NotImplementedError("MLP ensemble requires cell 1")


# ==========================================
# FT-TRANSFORMER (Gorishniy et al. 2021)
# ==========================================

def train_ft_transformer(X_train, y_train, X_test,
                         d_model=64, n_heads=4, ffn_dim=128,
                         n_blocks=3, dropout=0.1,
                         epochs=50, batch_size=256, lr=1e-3,
                         weight_decay=1e-4, seed=42, verbose=False):
    """
    FT-Transformer: Feature Tokenizer + [CLS] + Transformer Encoder.
    Per-feature learned embeddings, CLS token for readout.

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    class FTTransformer(nn.Module):
        def __init__(self, nf, d, nh, ffn, nblk, drop):
            super().__init__()
            self.feat_embeds = nn.ModuleList([nn.Linear(1, d) for _ in range(nf)])
            self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d, nhead=nh, dim_feedforward=ffn, dropout=drop,
                batch_first=True, activation='gelu')
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=nblk)
            self.norm = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)

        def forward(self, x):
            B = x.size(0)
            tokens = torch.stack([self.feat_embeds[i](x[:, i:i+1]) for i in range(x.size(1))], dim=1)
            tokens = torch.cat([self.cls_token.expand(B, -1, -1), tokens], dim=1)
            z = self.transformer(tokens)
            return self.head(self.norm(z[:, 0, :]))

    model = FTTransformer(nf, D, n_heads, ffn_dim, n_blocks, dropout).to(device)

    # Val split for early stopping (15%)
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    crit = nn.MSELoss()

    # LR warmup + cosine decay
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    warmup_epochs = max(1, epochs // 10)
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return ep / warmup_epochs
        progress = (ep - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    # Training with early stopping + grad clipping
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10

    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()

        # Val check
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xt_val), yt_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Restore best
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters())}


# ==========================================
# POLY PCA+OLS MODEL
# ==========================================

def train_poly_model(X_train, y_train, X_test,
                     n_components=200, d_model=64, ffn_dim=128,
                     epochs=50, batch_size=256, lr=1e-3,
                     dropout=0.1, weight_decay=1e-4,
                     seed=42, verbose=False):
    """
    Polynomial Cross-Feature Regression Transformer with PCA + OLS init.
    Replaces self-attention with explicit polynomial interaction features.

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    def poly_cross(x):
        B, S, Dm = x.shape
        per = torch.cat([x, x**2 - 1], dim=-1)
        pairs = torch.cat([x[:,i,:]*x[:,j,:] for i in range(S) for j in range(i,S)], dim=-1)
        return torch.cat([per, pairs.unsqueeze(1).expand(B,S,-1)], dim=-1)

    class FrozenPCA(nn.Module):
        def __init__(self, mean, components):
            super().__init__()
            self.register_buffer('mean', torch.from_numpy(mean.astype(np.float32)))
            self.register_buffer('comp', torch.from_numpy(components.astype(np.float32)))
            self.out_dim = components.shape[0]
        def forward(self, x): return (x - self.mean) @ self.comp.T

    class PolyPCAModel(nn.Module):
        def __init__(self, nf, pca_layer, d, ffn_dim, drop):
            super().__init__()
            self.pca = pca_layer
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            self.W = nn.Linear(pca_layer.out_dim, d)
            self.n1 = nn.LayerNorm(d)
            self.ffn = nn.Sequential(nn.Linear(d, ffn_dim), nn.ReLU(), nn.Dropout(drop),
                                     nn.Linear(ffn_dim, d), nn.Dropout(drop))
            self.n2 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)
        def embed_x(self, x): return self.embed(x.unsqueeze(-1)) + self.pos
        def get_poly(self, emb): return self.pca(poly_cross(emb))
        def forward(self, x):
            emb = self.embed_x(x); phi = self.get_poly(emb)
            z = self.n1(emb + self.W(phi))
            return self.head(self.n2(z + self.ffn(z)).mean(1))

    # Step 1: PCA on training poly features
    torch.manual_seed(seed)
    tmp_embed = nn.Linear(1, D); tmp_pos = nn.Parameter(torch.randn(1, nf, D)*0.02)
    Xt = torch.from_numpy(X_train.astype(np.float32))
    with torch.no_grad():
        emb = tmp_embed(Xt.unsqueeze(-1)) + tmp_pos
        phi_all = poly_cross(emb).reshape(-1, emb.shape[-1]*2 + D*nf*(nf+1)//2)
    n_comp = min(n_components, phi_all.shape[1], phi_all.shape[0])
    pca = PCA(n_components=n_comp, random_state=seed); pca.fit(phi_all.numpy())
    pca_layer = FrozenPCA(pca.mean_, pca.components_)

    # Step 2: Build model
    torch.manual_seed(seed)
    model = PolyPCAModel(nf, pca_layer, D, ffn_dim, dropout).to(device)

    # Step 3: OLS warm-start
    model.eval()
    with torch.no_grad():
        phi_pooled = model.get_poly(model.embed_x(Xt.to(device))).mean(1).cpu().numpy()
    reg = Ridge(alpha=1.0); reg.fit(phi_pooled, y_train)
    with torch.no_grad():
        model.W.weight.data[0] = torch.from_numpy(reg.coef_.astype(np.float32)).to(device)
        model.W.bias.data.zero_()
        model.head.weight.data.zero_(); model.head.weight.data[0,0] = 1.0
        model.head.bias.data.fill_(float(reg.intercept_))

    # Step 4: Train with early stopping + grad clipping (same as FTT for fairness)
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10

    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xt_val), yt_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # Step 5: Predict
    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters()), 'pca_components': n_comp}


# ==========================================
# DATASET LOADING ‚Äî VALIDATED
# ==========================================

OPENML_REGISTRY = {
    'Concrete':  (44959,  (1000, 1100),  8,  'Concrete compressive strength'),
    'Energy':    (41478,  (700, 800),     8,  'Energy efficiency'),
    'Kin8nm':    (189,    (8000, 8300),   8,  'Kin8nm robot arm'),
    'Protein':   (44963,  (45000, 46000), 9,  'Protein structure CASP'),
    'WineRed':   (40691,  (1500, 1700),   11, 'Wine quality red'),
    'Yacht':     (42370,  (300, 320),     6,  'Yacht hydrodynamics'),
    'Airfoil':   (44957,  (1400, 1600),   5,  'Airfoil self-noise'),
    'Abalone':   (183,    (4100, 4200),   7,  'Abalone age prediction'),
}


def load_datasets():
    enabled = [k for k, v in DATASETS.items() if v]
    print(f"\nüì¶ Loading {len(enabled)} datasets: {', '.join(enabled)}\n")
    loaded = {}

    if 'California' in enabled:
        try:
            d = fetch_california_housing()
            assert d.data.shape == (20640, 8)
            loaded['California'] = (d.data, d.target)
            print(f"  ‚úì California:  (20640, 8)")
        except Exception as e:
            print(f"  ‚úó California: {e}")

    if 'Diabetes' in enabled:
        try:
            d = load_diabetes()
            assert d.data.shape == (442, 10)
            loaded['Diabetes'] = (d.data, d.target)
            print(f"  ‚úì Diabetes:    (442, 10)")
        except Exception as e:
            print(f"  ‚úó Diabetes: {e}")

    for name in enabled:
        if name in ('California', 'Diabetes'):
            continue
        if name not in OPENML_REGISTRY:
            print(f"  ‚ö† {name}: not in registry, skipping")
            continue

        data_id, (n_min, n_max), expected_p, desc = OPENML_REGISTRY[name]
        try:
            data = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
            X = data.data.select_dtypes(include=[np.number]).values
            y_raw = data.target
            if y_raw is None:
                raise ValueError("Target column not set in OpenML metadata")
            if hasattr(y_raw, 'shape') and len(y_raw.shape) > 1 and y_raw.shape[1] > 1:
                y_raw = y_raw.iloc[:, 0]
            elif hasattr(y_raw, 'columns'):
                y_raw = y_raw.iloc[:, 0]
            if hasattr(y_raw, 'cat') and y_raw.dtype.name == 'category':
                y = y_raw.cat.codes.values.astype(float)
            else:
                y = np.array(y_raw).astype(float)
            mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
            X, y = X[mask], y[mask]
            N, P = X.shape
            if not (n_min <= N <= n_max):
                print(f"  ‚úó {name} (id={data_id}): N={N} outside expected range ({n_min}-{n_max})")
                continue
            if P != expected_p:
                print(f"  ‚úó {name} (id={data_id}): P={P} != expected {expected_p}")
                continue
            loaded[name] = (X, y)
            print(f"  ‚úì {name}:{' '*(10-len(name))}({N}, {P})  ‚Äî {desc}")
        except Exception as e:
            print(f"  ‚úó {name} (id={data_id}): {e}")

    print(f"\n‚Üí {len(loaded)}/{len(enabled)} datasets ready.\n")
    return loaded


# ==========================================
# BENCHMARK
# ==========================================

def run_benchmark():
    datasets = load_datasets()
    if not datasets:
        print("‚ùå No datasets loaded.")
        return None

    models_active = [m for m, on in [('OLS', RUN_OLS), ('RF', RUN_RF),
                     ('GBM', RUN_GBM), ('MLP', RUN_MLP), ('Attn', RUN_ATTN),
                     ('FTT', RUN_FTT), ('Poly', RUN_POLY)] if on]
    print(f"üîß Models: {', '.join(models_active)}")
    print(f"üîÅ Repeats: {N_REPEATS}, Test: {TEST_FRAC}, Max N: {MAX_N}\n")

    results = []
    t0 = time.time()

    for ds_name, (X_full, y_full) in datasets.items():
        N_full, P = X_full.shape

        if N_full > MAX_N:
            np.random.seed(SEED)
            idx = np.random.choice(N_full, MAX_N, replace=False)
            X_use, y_use = X_full[idx], y_full[idx]
            N_use = MAX_N
            label = f"{ds_name} (N={N_full}‚Üí{MAX_N}, P={P})"
        else:
            X_use, y_use = X_full, y_full
            N_use = N_full
            label = f"{ds_name} (N={N_use}, P={P})"

        print(f"{'='*70}\n  {label}\n{'='*70}")

        scores = {k: [] for k in models_active}
        times  = {k: [] for k in models_active}

        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            print(f"  Rep {r+1}/{N_REPEATS}...", end=" ", flush=True)

            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)

            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_train)
            X_te = scaler.transform(X_test)

            if 'OLS' in models_active:
                t_start = time.time()
                try:
                    m = LinearRegression().fit(X_tr, y_train)
                    scores['OLS'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['OLS'].append(np.nan)
                times['OLS'].append(time.time() - t_start)

            if 'RF' in models_active:
                t_start = time.time()
                try:
                    m = RandomForestRegressor(n_estimators=RF_TREES, max_features=RF_MTRY,
                                              n_jobs=-1, random_state=seed).fit(X_tr, y_train)
                    scores['RF'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['RF'].append(np.nan)
                times['RF'].append(time.time() - t_start)

            if 'GBM' in models_active:
                t_start = time.time()
                try:
                    m = GradientBoostingRegressor(n_estimators=GBM_TREES, learning_rate=GBM_LR,
                                                  random_state=seed).fit(X_tr, y_train)
                    scores['GBM'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['GBM'].append(np.nan)
                times['GBM'].append(time.time() - t_start)

            if 'MLP' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_mlp_ensemble(
                        X_tr, y_train, X_te, hidden_units=MLP_UNITS,
                        n_layers=MLP_LAYERS, dropout=MLP_DROP, ensemble_size=MLP_ENS,
                        max_epochs=MLP_EPOCHS, patience=MLP_PAT, batch_size=MLP_BS,
                        lr=MLP_LR_RATE, val_frac=MLP_VAL, seed=seed, verbose=False)
                    scores['MLP'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[MLP:{e}]", end=" ")
                    scores['MLP'].append(np.nan)
                times['MLP'].append(time.time() - t_start)

            if 'Attn' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_torch_model(
                        X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                        lr=ATTN_LR, reg_lambda=ATTN_REG, seed=seed, verbose=False)
                    r2_att = r2_score(y_test, y_pred)

                    if r2_att < 0.05:
                        y_pred, _ = train_torch_model(
                            X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                            lr=ATTN_LR, reg_lambda=ATTN_REG/100, seed=seed, verbose=False)
                        r2_att_retry = r2_score(y_test, y_pred)
                        if r2_att_retry > r2_att:
                            r2_att = r2_att_retry

                    if r2_att < 0:
                        y_var = np.var(y_test)
                        pred_var = np.var(y_pred)
                        pred_mean = np.mean(y_pred)
                        y_mean = np.mean(y_test)
                        mse = np.mean((y_pred - y_test)**2)
                        print(f"\n    ‚ö† ATTN DIAGNOSTIC: R¬≤={r2_att:.3f} | "
                              f"N_tr={len(y_train)} P={X_tr.shape[1]} | "
                              f"y_test: Œº={y_mean:.2f} œÉ¬≤={y_var:.2f} | "
                              f"≈∑: Œº={pred_mean:.2f} œÉ¬≤={pred_var:.2f} | "
                              f"MSE={mse:.2f}", end=" ")

                    scores['Attn'].append(r2_att)
                except Exception as e:
                    print(f"[Attn:{e}]", end=" ")
                    scores['Attn'].append(np.nan)
                times['Attn'].append(time.time() - t_start)

            if 'FTT' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_ft_transformer(
                        X_tr, y_train, X_te,
                        d_model=FTT_D_MODEL, n_heads=FTT_N_HEADS,
                        ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                        dropout=FTT_DROPOUT, epochs=FTT_EPOCHS,
                        batch_size=FTT_BS, lr=FTT_LR,
                        weight_decay=FTT_WD,
                        seed=seed, verbose=False)
                    scores['FTT'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[FTT:{e}]", end=" ")
                    scores['FTT'].append(np.nan)
                times['FTT'].append(time.time() - t_start)

            if 'Poly' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_poly_model(
                        X_tr, y_train, X_te,
                        n_components=POLY_PCA_COMP, d_model=POLY_D_MODEL,
                        ffn_dim=POLY_FFN_DIM, epochs=POLY_EPOCHS,
                        batch_size=POLY_BS, lr=POLY_LR,
                        dropout=POLY_DROPOUT, weight_decay=POLY_WD,
                        seed=seed, verbose=False)
                    scores['Poly'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[Poly:{e}]", end=" ")
                    scores['Poly'].append(np.nan)
                times['Poly'].append(time.time() - t_start)

            print("‚úì")

        avg = {k: np.nanmean(v) if v else np.nan for k, v in scores.items()}
        se  = {k: np.nanstd(v)  if v else np.nan for k, v in scores.items()}
        avg_t = {k: np.mean(v) if v else np.nan for k, v in times.items()}

        results.append({
            'Dataset': ds_name, 'N': N_use, 'P': P,
            **{m: avg[m] for m in models_active},
            **{f'{m}_se': se[m] for m in models_active},
            **{f'{m}_time': avg_t[m] for m in models_active},
        })

        parts = [f"{m}:{avg[m]:.3f}" for m in models_active if not np.isnan(avg[m])]
        valid = {k: v for k, v in avg.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '?'
        print(f"  ‚Üí {' | '.join(parts)}  |  Winner: {best}\n")

    # ==========================================
    # OUTPUT
    # ==========================================
    elapsed = time.time() - t0
    df = pd.DataFrame(results)

    print("\n" + "="*120)
    print(f"RESULTS ({elapsed/60:.1f} min, {N_REPEATS} reps, device={DEVICE})")
    print("="*120)

    header = f"{'Dataset':<12} {'N':>5} {'P':>3}"
    for m in models_active: header += f"  {m:>7}"
    header += "  |"
    for m in models_active: header += f"  {'t_'+m:>7}"
    header += "   Best"
    print(header)
    print("-"*120)

    for _, row in df.iterrows():
        line = f"{row['Dataset']:<12} {int(row['N']):>5} {int(row['P']):>3}"
        vals = {}
        for m in models_active:
            v = row[m]
            vals[m] = v
            line += f"  {v:>7.4f}" if not np.isnan(v) else f"  {'--':>7}"
        line += "  |"
        for m in models_active:
            t = row.get(f'{m}_time', np.nan)
            line += f"  {t:>6.1f}s" if not np.isnan(t) else f"  {'--':>7}"
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '--'
        line += f"   {best}"
        print(line)

    print("-"*120)

    # Win counts
    print("\nWin counts:")
    for m in models_active:
        wins = sum(1 for _, r in df.iterrows()
                   if not np.isnan(r[m]) and r[m] == max(r[c] for c in models_active if not np.isnan(r[c])))
        print(f"  {m}: {wins}/{len(df)}")

    # Average rank
    print("\nAverage rank (1=best):")
    ranks = df[models_active].rank(axis=1, ascending=False)
    for m in models_active:
        print(f"  {m}: {ranks[m].mean():.2f}")

    # Timing summary
    print("\nAvg time per rep (seconds):")
    for m in models_active:
        avg_t = df[f'{m}_time'].mean()
        print(f"  {m}: {avg_t:.1f}s")

    df.to_csv("real_data_results_v4.csv", index=False)
    print(f"\nüíæ Saved: real_data_results_v4.csv")

    # LaTeX
    col_names = {'OLS': 'OLS', 'RF': 'RF', 'GBM': 'GBM', 'MLP': 'MLP',
                 'Attn': r'Att.\ Reg.', 'FTT': 'FT-T', 'Poly': 'Poly'}
    print("\n\n% ========== LaTeX Table ==========")
    print(r"\begin{table}[htbp]")
    print(r"\centering")
    print(rf"\caption{{Out-of-sample $R^2$ on standard regression benchmarks ({N_REPEATS} random 80/20 splits, averaged). Best in bold.}}")
    print(r"\label{tab:real_data}")
    print(r"\begin{tabular}{lrr" + "c"*len(models_active) + "}")
    print(r"\toprule")
    hdr = r"Dataset & $N$ & $P$"
    for m in models_active: hdr += f" & {col_names.get(m, m)}"
    print(hdr + r" \\")
    print(r"\midrule")

    for _, row in df.iterrows():
        vals = {m: row[m] for m in models_active}
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best_val = max(valid.values()) if valid else None
        cells = []
        for m in models_active:
            v = row[m]
            if np.isnan(v): cells.append("--")
            elif best_val and abs(v - best_val) < 1e-4: cells.append(rf"\textbf{{{v:.3f}}}")
            else: cells.append(f"{v:.3f}")
        print(f"{row['Dataset']} & {int(row['N'])} & {int(row['P'])} & {' & '.join(cells)}" + r" \\")

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")

    return df


# ==========================================
# RUN
# ==========================================
df_results = run_benchmark()


Device: cuda
GPU: Tesla T4

üì¶ Loading 8 datasets: Yacht, Energy, Concrete, Airfoil, Abalone, California, Kin8nm, Protein

  ‚úì California:  (20640, 8)
  ‚úì Yacht:     (308, 6)  ‚Äî Yacht hydrodynamics
  ‚úì Energy:    (768, 8)  ‚Äî Energy efficiency
  ‚úì Concrete:  (1030, 8)  ‚Äî Concrete compressive strength
  ‚úì Airfoil:   (1503, 5)  ‚Äî Airfoil self-noise
  ‚úì Abalone:   (4177, 7)  ‚Äî Abalone age prediction
  ‚úì Kin8nm:    (8192, 8)  ‚Äî Kin8nm robot arm
  ‚úì Protein:   (45730, 9)  ‚Äî Protein structure CASP

‚Üí 8/8 datasets ready.

üîß Models: OLS, RF, Attn, FTT, Poly
üîÅ Repeats: 5, Test: 0.2, Max N: 5000

  California (N=20640‚Üí5000, P=8)
  Rep 1/5... ‚úì
  Rep 2/5... ‚úì
  Rep 3/5... ‚úì
  Rep 4/5... ‚úì
  Rep 5/5... ‚úì
  ‚Üí OLS:0.597 | RF:0.777 | Attn:0.715 | FTT:0.771 | Poly:0.769  |  Winner: RF

  Yacht (N=308, P=6)
  Rep 1/5... ‚úì
  Rep 2/5... ‚úì
  Rep 3/5... 
    ‚ö† ATTN DIAGNOSTIC: R¬≤=-0.004 | N_tr=246 P=6 | y_test: Œº=8.87 œÉ¬≤=144.97 | ≈∑: Œº=8.08 œÉ

In [5]:
"""
========================================================
REAL DATA BENCHMARK FOR ATTENTION REGRESSION ‚Äî v4
With Polynomial Cross-Feature Transformer (Poly PCA+OLS)

Paste in a NEW Colab cell after running simulation cell.
Run cell 1 first (even briefly) to load function defs.
========================================================
"""

# ==========================================
# CONFIG ‚Äî EDIT THIS SECTION
# ==========================================

DATASETS = {
    # --- Small & fast ---
    'Diabetes':    False,       # 442 obs, 10 feat (sklearn built-in)
    'Yacht':       True,       # 308 obs, 6 feat
    'Energy':      True,       # 768 obs, 8 feat
    # --- Medium ---
    'Concrete':    True,       # 1030 obs, 8 feat (nonlinear!)
    'Airfoil':     True,       # 1503 obs, 5 feat
    'WineRed':     False,       # 1599 obs, 11 feat
    # --- Larger ---
    'Abalone':     True,       # 4177 obs, 8 feat
    'California':  True,       # 20640‚Üí5000 obs, 8 feat (sklearn built-in)
    'Kin8nm':      True,       # 8192‚Üí5000 obs, 8 feat
    'Protein':     True,       # 45730‚Üí5000 obs, 9 feat
}

N_REPEATS = 5
TEST_FRAC = 0.2
MAX_N = 5000
SEED = 42

# Models
RUN_OLS  = True
RUN_RF   = True
RUN_GBM  = False
RUN_MLP  = False
RUN_ATTN = False
RUN_FTT  = False
RUN_POLY = True

# Hyperparameters
RF_TREES = 500;  RF_MTRY = 1/3
GBM_TREES = 500; GBM_LR = 0.01
MLP_LAYERS = 3;  MLP_UNITS = 200; MLP_DROP = 0.2
MLP_ENS = 1;    MLP_EPOCHS = 1000; MLP_PAT = 20
MLP_BS = 64;     MLP_LR_RATE = 1e-3; MLP_VAL = 0.15
ATTN_M = 4;      ATTN_STEPS = 300; ATTN_LR = 1e-3; ATTN_REG = 1e-3

# FT-Transformer hyperparameters (Gorishniy et al. 2021)
FTT_D_MODEL = 64
FTT_N_HEADS = 4
FTT_FFN_DIM = 128
FTT_N_BLOCKS = 3
FTT_DROPOUT = 0.1
FTT_EPOCHS = 200
FTT_BS = 256
FTT_LR = 1e-3
FTT_WD = 1e-4

# Poly Raw hyperparameters (no PCA, no embedding before poly)
POLY_D_MODEL = 64
POLY_FFN_DIM = 128
POLY_EPOCHS = 200
POLY_BS = 256
POLY_LR = 1e-3
POLY_DROPOUT = 0.1
POLY_WD = 1e-4

# ==========================================
# END CONFIG
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing, load_diabetes, fetch_openml
from sklearn.decomposition import PCA
import warnings, time
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ==========================================
# ATTENTION REGRESSION (simple 2-block)
# Self-contained ‚Äî no dependency on cell 1
# ==========================================

def train_torch_model(X_train, y_train, X_test,
                      M=4, steps=300, lr=1e-3,
                      reg_lambda=1e-3, seed=42, verbose=False):
    """
    Simple attention-based regression: embed features as tokens,
    2-block multi-head self-attention, pool, linear head.
    Trains for `steps` gradient steps (not epochs).
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = 64

    class AttnRegressor(nn.Module):
        def __init__(self, nf, d, nh):
            super().__init__()
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            self.attn1 = nn.MultiheadAttention(d, nh, batch_first=True, dropout=0.1)
            self.n1 = nn.LayerNorm(d)
            self.ff1 = nn.Sequential(nn.Linear(d, d*2), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d*2, d))
            self.n2 = nn.LayerNorm(d)
            self.attn2 = nn.MultiheadAttention(d, nh, batch_first=True, dropout=0.1)
            self.n3 = nn.LayerNorm(d)
            self.ff2 = nn.Sequential(nn.Linear(d, d*2), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d*2, d))
            self.n4 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)

        def forward(self, x):
            x = self.embed(x.unsqueeze(-1)) + self.pos
            a, _ = self.attn1(x, x, x); x = self.n1(x + a); x = self.n2(x + self.ff1(x))
            a, _ = self.attn2(x, x, x); x = self.n3(x + a); x = self.n4(x + self.ff2(x))
            return self.head(x.mean(1))

    model = AttnRegressor(nf, D, max(1, M)).to(device)

    # Val split for early stopping
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=256, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=reg_lambda)
    crit = nn.MSELoss()

    best_val = float('inf'); best_state = None; patience = 0; step = 0
    done = False
    while not done:
        model.train()
        for Xb, yb in tr_loader:
            if step >= steps:
                done = True; break
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); step += 1

        # Val check each pass through data
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xt_val), yt_val).item()
        if vl < best_val:
            best_val = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= 10: break

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters())}


def train_mlp_ensemble(*args, **kwargs):
    """Stub ‚Äî not implemented in standalone mode."""
    raise NotImplementedError("MLP ensemble requires cell 1")


# ==========================================
# FT-TRANSFORMER (Gorishniy et al. 2021)
# ==========================================

def train_ft_transformer(X_train, y_train, X_test,
                         d_model=64, n_heads=4, ffn_dim=128,
                         n_blocks=3, dropout=0.1,
                         epochs=50, batch_size=256, lr=1e-3,
                         weight_decay=1e-4, seed=42, verbose=False):
    """
    FT-Transformer: Feature Tokenizer + [CLS] + Transformer Encoder.
    Per-feature learned embeddings, CLS token for readout.

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    class FTTransformer(nn.Module):
        def __init__(self, nf, d, nh, ffn, nblk, drop):
            super().__init__()
            self.feat_embeds = nn.ModuleList([nn.Linear(1, d) for _ in range(nf)])
            self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d, nhead=nh, dim_feedforward=ffn, dropout=drop,
                batch_first=True, activation='gelu')
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=nblk)
            self.norm = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)

        def forward(self, x):
            B = x.size(0)
            tokens = torch.stack([self.feat_embeds[i](x[:, i:i+1]) for i in range(x.size(1))], dim=1)
            tokens = torch.cat([self.cls_token.expand(B, -1, -1), tokens], dim=1)
            z = self.transformer(tokens)
            return self.head(self.norm(z[:, 0, :]))

    model = FTTransformer(nf, D, n_heads, ffn_dim, n_blocks, dropout).to(device)

    # Val split for early stopping (15%)
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    crit = nn.MSELoss()

    # LR warmup + cosine decay
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    warmup_epochs = max(1, epochs // 10)
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return ep / warmup_epochs
        progress = (ep - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    # Training with early stopping + grad clipping
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10

    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()

        # Val check
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xt_val), yt_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Restore best
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters())}


# ==========================================
# POLY RAW MODEL
# ==========================================
# Raw polynomial features of x (no embedding, no PCA).
# œÜ(x) = [x, x¬≤-1, x_i¬∑x_j for all i‚â§j] ‚Üí W ‚Üí LN ‚Üí FFN ‚Üí head ‚Üí ≈∑
# All parameters trained end-to-end. No preprocessing. No frozen matrices.

def train_poly_model(X_train, y_train, X_test,
                     d_model=64, ffn_dim=128,
                     epochs=200, batch_size=256, lr=1e-3,
                     dropout=0.1, weight_decay=1e-4,
                     seed=42, verbose=False, **kwargs):
    """
    Polynomial Cross-Feature Regression: replaces attention with explicit
    degree-2 polynomial interactions on raw features.

    œÜ(x) = [x‚ÇÅ,...,x‚Çö, x‚ÇÅ¬≤-1,...,x‚Çö¬≤-1, x‚ÇÅx‚ÇÇ, x‚ÇÅx‚ÇÉ,...,x‚Çö‚Çã‚ÇÅx‚Çö]
    ≈∑ = head(LN(FFN(LN(W¬∑œÜ(x)))))

    Interface matches train_torch_model / train_mlp_ensemble:
      Args: X_train (N,P), y_train (N,), X_test (M,P), **hyperparams
      Returns: (y_pred (M,), info_dict)
    """
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model
    n_pairs = nf * (nf + 1) // 2
    poly_dim = nf + nf + n_pairs  # main + squared + cross

    def raw_poly(x):
        """x: (B, P) ‚Üí (B, P + P + P(P+1)/2)"""
        B, P = x.shape
        main = x
        sq = x**2 - 1
        cross = [x[:, i] * x[:, j] for i in range(P) for j in range(i, P)]
        return torch.cat([main, sq, torch.stack(cross, 1)], 1)

    class PolyRawModel(nn.Module):
        def __init__(self, poly_dim, d, ffn_dim, drop):
            super().__init__()
            self.W = nn.Linear(poly_dim, d)
            self.n1 = nn.LayerNorm(d)
            self.ffn = nn.Sequential(nn.Linear(d, ffn_dim), nn.ReLU(), nn.Dropout(drop),
                                     nn.Linear(ffn_dim, d), nn.Dropout(drop))
            self.n2 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)
        def forward(self, x):
            z = self.n1(self.W(raw_poly(x)))
            return self.head(self.n2(z + self.ffn(z)))

    model = PolyRawModel(poly_dim, D, ffn_dim, dropout).to(device)

    # Val split for early stopping (15%)
    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    patience = 10

    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xt_val), yt_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X_test.astype(np.float32)).to(device)).cpu().numpy().ravel()
    return y_pred, {'n_params': sum(p.numel() for p in model.parameters()), 'poly_dim': poly_dim}


# ==========================================
# DATASET LOADING ‚Äî VALIDATED
# ==========================================

OPENML_REGISTRY = {
    'Concrete':  (44959,  (1000, 1100),  8,  'Concrete compressive strength'),
    'Energy':    (41478,  (700, 800),     8,  'Energy efficiency'),
    'Kin8nm':    (189,    (8000, 8300),   8,  'Kin8nm robot arm'),
    'Protein':   (44963,  (45000, 46000), 9,  'Protein structure CASP'),
    'WineRed':   (40691,  (1500, 1700),   11, 'Wine quality red'),
    'Yacht':     (42370,  (300, 320),     6,  'Yacht hydrodynamics'),
    'Airfoil':   (44957,  (1400, 1600),   5,  'Airfoil self-noise'),
    'Abalone':   (183,    (4100, 4200),   7,  'Abalone age prediction'),
}


def load_datasets():
    enabled = [k for k, v in DATASETS.items() if v]
    print(f"\nüì¶ Loading {len(enabled)} datasets: {', '.join(enabled)}\n")
    loaded = {}

    if 'California' in enabled:
        try:
            d = fetch_california_housing()
            assert d.data.shape == (20640, 8)
            loaded['California'] = (d.data, d.target)
            print(f"  ‚úì California:  (20640, 8)")
        except Exception as e:
            print(f"  ‚úó California: {e}")

    if 'Diabetes' in enabled:
        try:
            d = load_diabetes()
            assert d.data.shape == (442, 10)
            loaded['Diabetes'] = (d.data, d.target)
            print(f"  ‚úì Diabetes:    (442, 10)")
        except Exception as e:
            print(f"  ‚úó Diabetes: {e}")

    for name in enabled:
        if name in ('California', 'Diabetes'):
            continue
        if name not in OPENML_REGISTRY:
            print(f"  ‚ö† {name}: not in registry, skipping")
            continue

        data_id, (n_min, n_max), expected_p, desc = OPENML_REGISTRY[name]
        try:
            data = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
            X = data.data.select_dtypes(include=[np.number]).values
            y_raw = data.target
            if y_raw is None:
                raise ValueError("Target column not set in OpenML metadata")
            if hasattr(y_raw, 'shape') and len(y_raw.shape) > 1 and y_raw.shape[1] > 1:
                y_raw = y_raw.iloc[:, 0]
            elif hasattr(y_raw, 'columns'):
                y_raw = y_raw.iloc[:, 0]
            if hasattr(y_raw, 'cat') and y_raw.dtype.name == 'category':
                y = y_raw.cat.codes.values.astype(float)
            else:
                y = np.array(y_raw).astype(float)
            mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
            X, y = X[mask], y[mask]
            N, P = X.shape
            if not (n_min <= N <= n_max):
                print(f"  ‚úó {name} (id={data_id}): N={N} outside expected range ({n_min}-{n_max})")
                continue
            if P != expected_p:
                print(f"  ‚úó {name} (id={data_id}): P={P} != expected {expected_p}")
                continue
            loaded[name] = (X, y)
            print(f"  ‚úì {name}:{' '*(10-len(name))}({N}, {P})  ‚Äî {desc}")
        except Exception as e:
            print(f"  ‚úó {name} (id={data_id}): {e}")

    print(f"\n‚Üí {len(loaded)}/{len(enabled)} datasets ready.\n")
    return loaded


# ==========================================
# BENCHMARK
# ==========================================

def run_benchmark():
    datasets = load_datasets()
    if not datasets:
        print("‚ùå No datasets loaded.")
        return None

    models_active = [m for m, on in [('OLS', RUN_OLS), ('RF', RUN_RF),
                     ('GBM', RUN_GBM), ('MLP', RUN_MLP), ('Attn', RUN_ATTN),
                     ('FTT', RUN_FTT), ('Poly', RUN_POLY)] if on]
    print(f"üîß Models: {', '.join(models_active)}")
    print(f"üîÅ Repeats: {N_REPEATS}, Test: {TEST_FRAC}, Max N: {MAX_N}\n")

    results = []
    t0 = time.time()

    for ds_name, (X_full, y_full) in datasets.items():
        N_full, P = X_full.shape

        if N_full > MAX_N:
            np.random.seed(SEED)
            idx = np.random.choice(N_full, MAX_N, replace=False)
            X_use, y_use = X_full[idx], y_full[idx]
            N_use = MAX_N
            label = f"{ds_name} (N={N_full}‚Üí{MAX_N}, P={P})"
        else:
            X_use, y_use = X_full, y_full
            N_use = N_full
            label = f"{ds_name} (N={N_use}, P={P})"

        print(f"{'='*70}\n  {label}\n{'='*70}")

        scores = {k: [] for k in models_active}
        times  = {k: [] for k in models_active}

        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            print(f"  Rep {r+1}/{N_REPEATS}...", end=" ", flush=True)

            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)

            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_train)
            X_te = scaler.transform(X_test)

            if 'OLS' in models_active:
                t_start = time.time()
                try:
                    m = LinearRegression().fit(X_tr, y_train)
                    scores['OLS'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['OLS'].append(np.nan)
                times['OLS'].append(time.time() - t_start)

            if 'RF' in models_active:
                t_start = time.time()
                try:
                    m = RandomForestRegressor(n_estimators=RF_TREES, max_features=RF_MTRY,
                                              n_jobs=-1, random_state=seed).fit(X_tr, y_train)
                    scores['RF'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['RF'].append(np.nan)
                times['RF'].append(time.time() - t_start)

            if 'GBM' in models_active:
                t_start = time.time()
                try:
                    m = GradientBoostingRegressor(n_estimators=GBM_TREES, learning_rate=GBM_LR,
                                                  random_state=seed).fit(X_tr, y_train)
                    scores['GBM'].append(r2_score(y_test, m.predict(X_te)))
                except: scores['GBM'].append(np.nan)
                times['GBM'].append(time.time() - t_start)

            if 'MLP' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_mlp_ensemble(
                        X_tr, y_train, X_te, hidden_units=MLP_UNITS,
                        n_layers=MLP_LAYERS, dropout=MLP_DROP, ensemble_size=MLP_ENS,
                        max_epochs=MLP_EPOCHS, patience=MLP_PAT, batch_size=MLP_BS,
                        lr=MLP_LR_RATE, val_frac=MLP_VAL, seed=seed, verbose=False)
                    scores['MLP'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[MLP:{e}]", end=" ")
                    scores['MLP'].append(np.nan)
                times['MLP'].append(time.time() - t_start)

            if 'Attn' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_torch_model(
                        X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                        lr=ATTN_LR, reg_lambda=ATTN_REG, seed=seed, verbose=False)
                    r2_att = r2_score(y_test, y_pred)

                    if r2_att < 0.05:
                        y_pred, _ = train_torch_model(
                            X_tr, y_train, X_te, M=ATTN_M, steps=ATTN_STEPS,
                            lr=ATTN_LR, reg_lambda=ATTN_REG/100, seed=seed, verbose=False)
                        r2_att_retry = r2_score(y_test, y_pred)
                        if r2_att_retry > r2_att:
                            r2_att = r2_att_retry

                    if r2_att < 0:
                        y_var = np.var(y_test)
                        pred_var = np.var(y_pred)
                        pred_mean = np.mean(y_pred)
                        y_mean = np.mean(y_test)
                        mse = np.mean((y_pred - y_test)**2)
                        print(f"\n    ‚ö† ATTN DIAGNOSTIC: R¬≤={r2_att:.3f} | "
                              f"N_tr={len(y_train)} P={X_tr.shape[1]} | "
                              f"y_test: Œº={y_mean:.2f} œÉ¬≤={y_var:.2f} | "
                              f"≈∑: Œº={pred_mean:.2f} œÉ¬≤={pred_var:.2f} | "
                              f"MSE={mse:.2f}", end=" ")

                    scores['Attn'].append(r2_att)
                except Exception as e:
                    print(f"[Attn:{e}]", end=" ")
                    scores['Attn'].append(np.nan)
                times['Attn'].append(time.time() - t_start)

            if 'FTT' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_ft_transformer(
                        X_tr, y_train, X_te,
                        d_model=FTT_D_MODEL, n_heads=FTT_N_HEADS,
                        ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                        dropout=FTT_DROPOUT, epochs=FTT_EPOCHS,
                        batch_size=FTT_BS, lr=FTT_LR,
                        weight_decay=FTT_WD,
                        seed=seed, verbose=False)
                    scores['FTT'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[FTT:{e}]", end=" ")
                    scores['FTT'].append(np.nan)
                times['FTT'].append(time.time() - t_start)

            if 'Poly' in models_active:
                t_start = time.time()
                try:
                    y_pred, _ = train_poly_model(
                        X_tr, y_train, X_te,
                        d_model=POLY_D_MODEL,
                        ffn_dim=POLY_FFN_DIM, epochs=POLY_EPOCHS,
                        batch_size=POLY_BS, lr=POLY_LR,
                        dropout=POLY_DROPOUT, weight_decay=POLY_WD,
                        seed=seed, verbose=False)
                    scores['Poly'].append(r2_score(y_test, y_pred))
                except Exception as e:
                    print(f"[Poly:{e}]", end=" ")
                    scores['Poly'].append(np.nan)
                times['Poly'].append(time.time() - t_start)

            print("‚úì")

        avg = {k: np.nanmean(v) if v else np.nan for k, v in scores.items()}
        se  = {k: np.nanstd(v)  if v else np.nan for k, v in scores.items()}
        avg_t = {k: np.mean(v) if v else np.nan for k, v in times.items()}

        results.append({
            'Dataset': ds_name, 'N': N_use, 'P': P,
            **{m: avg[m] for m in models_active},
            **{f'{m}_se': se[m] for m in models_active},
            **{f'{m}_time': avg_t[m] for m in models_active},
        })

        parts = [f"{m}:{avg[m]:.3f}" for m in models_active if not np.isnan(avg[m])]
        valid = {k: v for k, v in avg.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '?'
        print(f"  ‚Üí {' | '.join(parts)}  |  Winner: {best}\n")

    # ==========================================
    # OUTPUT
    # ==========================================
    elapsed = time.time() - t0
    df = pd.DataFrame(results)

    print("\n" + "="*120)
    print(f"RESULTS ({elapsed/60:.1f} min, {N_REPEATS} reps, device={DEVICE})")
    print("="*120)

    header = f"{'Dataset':<12} {'N':>5} {'P':>3}"
    for m in models_active: header += f"  {m:>7}"
    header += "  |"
    for m in models_active: header += f"  {'t_'+m:>7}"
    header += "   Best"
    print(header)
    print("-"*120)

    for _, row in df.iterrows():
        line = f"{row['Dataset']:<12} {int(row['N']):>5} {int(row['P']):>3}"
        vals = {}
        for m in models_active:
            v = row[m]
            vals[m] = v
            line += f"  {v:>7.4f}" if not np.isnan(v) else f"  {'--':>7}"
        line += "  |"
        for m in models_active:
            t = row.get(f'{m}_time', np.nan)
            line += f"  {t:>6.1f}s" if not np.isnan(t) else f"  {'--':>7}"
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best = max(valid, key=valid.get) if valid else '--'
        line += f"   {best}"
        print(line)

    print("-"*120)

    # Win counts
    print("\nWin counts:")
    for m in models_active:
        wins = sum(1 for _, r in df.iterrows()
                   if not np.isnan(r[m]) and r[m] == max(r[c] for c in models_active if not np.isnan(r[c])))
        print(f"  {m}: {wins}/{len(df)}")

    # Average rank
    print("\nAverage rank (1=best):")
    ranks = df[models_active].rank(axis=1, ascending=False)
    for m in models_active:
        print(f"  {m}: {ranks[m].mean():.2f}")

    # Timing summary
    print("\nAvg time per rep (seconds):")
    for m in models_active:
        avg_t = df[f'{m}_time'].mean()
        print(f"  {m}: {avg_t:.1f}s")

    df.to_csv("real_data_results_v4.csv", index=False)
    print(f"\nüíæ Saved: real_data_results_v4.csv")

    # LaTeX
    col_names = {'OLS': 'OLS', 'RF': 'RF', 'GBM': 'GBM', 'MLP': 'MLP',
                 'Attn': r'Att.\ Reg.', 'FTT': 'FT-T', 'Poly': 'Poly'}
    print("\n\n% ========== LaTeX Table ==========")
    print(r"\begin{table}[htbp]")
    print(r"\centering")
    print(rf"\caption{{Out-of-sample $R^2$ on standard regression benchmarks ({N_REPEATS} random 80/20 splits, averaged). Best in bold.}}")
    print(r"\label{tab:real_data}")
    print(r"\begin{tabular}{lrr" + "c"*len(models_active) + "}")
    print(r"\toprule")
    hdr = r"Dataset & $N$ & $P$"
    for m in models_active: hdr += f" & {col_names.get(m, m)}"
    print(hdr + r" \\")
    print(r"\midrule")

    for _, row in df.iterrows():
        vals = {m: row[m] for m in models_active}
        valid = {k: v for k, v in vals.items() if not np.isnan(v)}
        best_val = max(valid.values()) if valid else None
        cells = []
        for m in models_active:
            v = row[m]
            if np.isnan(v): cells.append("--")
            elif best_val and abs(v - best_val) < 1e-4: cells.append(rf"\textbf{{{v:.3f}}}")
            else: cells.append(f"{v:.3f}")
        print(f"{row['Dataset']} & {int(row['N'])} & {int(row['P'])} & {' & '.join(cells)}" + r" \\")

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")

    return df


# ==========================================
# RUN
# ==========================================
#df_results = run_benchmark()


Device: cuda
GPU: NVIDIA L4


In [6]:
"""
============================================================
ONE-RUN REBUTTAL EXPERIMENTS  (NeurIPS 31482)  --  v2
Paste in a NEW Colab cell AFTER running the v4 benchmark cell
(needs train_ft_transformer, load_datasets, config constants).
GPU runtime. Rough budget on a T4: Part 1 ~60-90 min,
Part 2 ~30-60 min, Part 3 ~5 min.
============================================================
PART 1  Ablation grid A0-A5+RB          (xZ71 W3/Q3, meta-review)
PART 2  Uncapped N: California/Kin8nm/Protein, OLS/RF/FT-T/RB
                                        (xZ71 W2/Q2)
PART 3  TabPFN reference column, 8 capped datasets   (Ayqz Q2)

Every rep's R2 is printed. Per-dataset mean and sd printed.
Final consolidated report reprints all tables + verification vs
Table 1 + OpenReview markdown + LaTeX. CSVs saved per part.
NOT run here (on purpose): Att.Reg at full N -- this notebook's
standalone train_torch_model is not the pipeline behind the
paper's Att.Reg column and produces inconsistent numbers.
"""

RUN_ABLATION = True
RUN_UNCAPPED = True
RUN_TABPFN   = True

import numpy as np, pandas as pd, torch, torch.nn as nn, time, warnings, sys, subprocess
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')

PAPER = {
    'California': {'A0': 0.771, 'RB': 0.769}, 'Yacht':    {'A0': 0.445, 'RB': 0.960},
    'Energy':     {'A0': 0.992, 'RB': 0.995}, 'Concrete': {'A0': 0.652, 'RB': 0.892},
    'Airfoil':    {'A0': -75.451, 'RB': 0.927}, 'Abalone': {'A0': 0.314, 'RB': 0.331},
    'Kin8nm':     {'A0': 0.909, 'RB': 0.917}, 'Protein':  {'A0': 0.418, 'RB': 0.453},
}
UNCAPPED = ['California', 'Kin8nm', 'Protein']
ABL = ['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'RB']
LBL = {'A0': 'FT-Transformer (paper)', 'A1': 'FT-T + polynomial inputs',
       'A2': 'RB w/o polynomial expansion', 'A3': 'RB w/o PCA compression',
       'A4': 'RB skeleton, softmax instead of regression',
       'A5': 'FT-T parameter-matched (~30K)', 'RB': 'Regression Block (paper)'}

# ============================================================
# RB VARIANT TRAINER (exact mirror of train_poly_model with
# flags; pca_fit_cap added for full-N feasibility, see note)
# ============================================================
def train_rb_variant(X_train, y_train, X_test,
                     use_poly=True, use_pca=True, softmax_mixer=False,
                     n_components=POLY_PCA_COMP, d_model=POLY_D_MODEL,
                     ffn_dim=POLY_FFN_DIM, epochs=POLY_EPOCHS,
                     batch_size=POLY_BS, lr=POLY_LR, dropout=POLY_DROPOUT,
                     weight_decay=POLY_WD, n_heads=4, seed=42,
                     pca_fit_cap=5000):
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE; nf = X_train.shape[1]; D = d_model

    def poly_cross(x):
        B, S, Dm = x.shape
        per = torch.cat([x, x**2 - 1], dim=-1)
        pairs = torch.cat([x[:, i, :] * x[:, j, :] for i in range(S) for j in range(i, S)], dim=-1)
        return torch.cat([per, pairs.unsqueeze(1).expand(B, S, -1)], dim=-1)

    def features(x): return poly_cross(x) if use_poly else x

    class FrozenPCA(nn.Module):
        def __init__(self, mean, components):
            super().__init__()
            self.register_buffer('mean', torch.from_numpy(mean.astype(np.float32)))
            self.register_buffer('comp', torch.from_numpy(components.astype(np.float32)))
            self.out_dim = components.shape[0]
        def forward(self, x): return (x - self.mean) @ self.comp.T

    class Identity(nn.Module):
        def __init__(self, dim): super().__init__(); self.out_dim = dim
        def forward(self, x): return x

    class RBVariant(nn.Module):
        def __init__(self, nf, red_layer, d, ffn_dim, drop):
            super().__init__()
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            if softmax_mixer:
                self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True, dropout=drop)
            else:
                self.red = red_layer
                self.W = nn.Linear(red_layer.out_dim, d)
            self.n1 = nn.LayerNorm(d)
            self.ffn = nn.Sequential(nn.Linear(d, ffn_dim), nn.ReLU(), nn.Dropout(drop),
                                     nn.Linear(ffn_dim, d), nn.Dropout(drop))
            self.n2 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)
        def embed_x(self, x): return self.embed(x.unsqueeze(-1)) + self.pos
        def get_feat(self, emb): return self.red(features(emb))
        def forward(self, x):
            emb = self.embed_x(x)
            if softmax_mixer:
                a, _ = self.attn(emb, emb, emb)
                z = self.n1(emb + a)
            else:
                z = self.n1(emb + self.W(self.get_feat(emb)))
            return self.head(self.n2(z + self.ffn(z)).mean(1))

    red_layer = None
    if not softmax_mixer:
        torch.manual_seed(seed)
        tmp_embed = nn.Linear(1, D); tmp_pos = nn.Parameter(torch.randn(1, nf, D) * 0.02)
        # PCA fit subsample at full N (frozen preprocessing; only deviation, noted in caption)
        if len(X_train) > pca_fit_cap:
            sub = np.random.RandomState(seed).choice(len(X_train), pca_fit_cap, replace=False)
            X_pca = X_train[sub]
        else:
            X_pca = X_train
        Xt = torch.from_numpy(X_pca.astype(np.float32))
        with torch.no_grad():
            emb = tmp_embed(Xt.unsqueeze(-1)) + tmp_pos
            phi_all = features(emb)
            feat_dim = phi_all.shape[-1]
            phi_all = phi_all.reshape(-1, feat_dim)
        if use_pca:
            n_comp = min(n_components, phi_all.shape[1], phi_all.shape[0])
            p = PCA(n_components=n_comp, random_state=seed); p.fit(phi_all.numpy())
            red_layer = FrozenPCA(p.mean_, p.components_)
        else:
            red_layer = Identity(feat_dim)

    torch.manual_seed(seed)
    model = RBVariant(nf, red_layer, D, ffn_dim, dropout).to(device)

    if not softmax_mixer:  # ridge warm start, exactly as the original
        model.eval()
        WS_N = min(len(X_train), 20000)
        Xt = torch.from_numpy(X_train[:WS_N].astype(np.float32)).to(device)
        with torch.no_grad():
            phi_pooled = model.get_feat(model.embed_x(Xt)).mean(1).cpu().numpy()
        reg = Ridge(alpha=1.0); reg.fit(phi_pooled, y_train[:WS_N])
        with torch.no_grad():
            model.W.weight.data[0] = torch.from_numpy(reg.coef_.astype(np.float32)).to(device)
            model.W.bias.data.zero_()
            model.head.weight.data.zero_(); model.head.weight.data[0, 0] = 1.0
            model.head.bias.data.fill_(float(reg.intercept_))

    N = X_train.shape[0]; n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)
    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()
    best, best_state, pat = float('inf'), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xt_val), yt_val).item()
        if vl < best:
            best, best_state, pat = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= 10: break
    if best_state: model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test), 4096):
            xb = torch.from_numpy(X_test[i:i+4096].astype(np.float32)).to(device)
            preds.append(model(xb).cpu().numpy().ravel())
    return np.concatenate(preds), {'n_params': sum(p.numel() for p in model.parameters() if p.requires_grad)}

# ============================================================
# PARAM COUNTS + A5 SEARCH
# ============================================================
def ftt_param_count(nf, d, nh, ffn, nblk):
    m = nn.ModuleDict({
        'emb': nn.ModuleList([nn.Linear(1, d) for _ in range(nf)]),
        'enc': nn.TransformerEncoder(nn.TransformerEncoderLayer(
            d_model=d, nhead=nh, dim_feedforward=ffn, dropout=0.1,
            batch_first=True, activation='gelu'), num_layers=nblk),
        'norm': nn.LayerNorm(d), 'head': nn.Linear(d, 1)})
    return sum(p.numel() for p in m.parameters()) + d

def rb_param_count(nf, d=POLY_D_MODEL, ffn=POLY_FFN_DIM, ncomp=POLY_PCA_COMP):
    m = nn.ModuleDict({'e': nn.Linear(1, d), 'W': nn.Linear(ncomp, d),
                       'n1': nn.LayerNorm(d),
                       'f': nn.Sequential(nn.Linear(d, ffn), nn.Linear(ffn, d)),
                       'n2': nn.LayerNorm(d), 'h': nn.Linear(d, 1)})
    return sum(p.numel() for p in m.parameters()) + nf * d

def pick_a5_dmodel(nf, target):
    best_d, best_gap = None, float('inf')
    for d in [16, 20, 24, 28, 32, 36, 40, 48]:
        if d % 4: continue
        gap = abs(ftt_param_count(nf, d, 4, 2 * d, FTT_N_BLOCKS) - target)
        if gap < best_gap: best_d, best_gap = d, gap
    return best_d

def reps_str(v): return " ".join(f"{x:+.3f}" for x in v)

datasets = load_datasets()
ablation_rows, uncapped_rows, tabpfn_rows = [], [], []
T0 = time.time()

# ============================================================
# PART 1: ABLATION GRID
# ============================================================
if RUN_ABLATION:
    print("\n" + "#" * 78 + "\n# PART 1: ABLATION GRID (A0-A5 + RB)\n" + "#" * 78)
    for ds_name, (X_full, y_full) in datasets.items():
        N_full, P = X_full.shape
        if N_full > MAX_N:
            np.random.seed(SEED)
            idx = np.random.choice(N_full, MAX_N, replace=False)
            X_use, y_use = X_full[idx], y_full[idx]
        else:
            X_use, y_use = X_full, y_full
        rb_target = rb_param_count(P)
        a5_d = pick_a5_dmodel(P, rb_target)
        print(f"\n=== {ds_name} (N={len(y_use)}, P={P}) | RB params ~{rb_target:,} "
              f"-> A5 d_model={a5_d} (~{ftt_param_count(P, a5_d, 4, 2*a5_d, FTT_N_BLOCKS):,}) ===")
        scores = {c: [] for c in ABL}; params = {c: None for c in ABL}
        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)
            sc = StandardScaler(); X_tr = sc.fit_transform(X_train); X_te = sc.transform(X_test)
            pf = PolynomialFeatures(degree=2, include_bias=False)
            X_tr_pf = pf.fit_transform(X_tr); X_te_pf = pf.transform(X_te)
            sc2 = StandardScaler(); X_tr_pf = sc2.fit_transform(X_tr_pf); X_te_pf = sc2.transform(X_te_pf)
            jobs = {
                'A0': lambda: train_ft_transformer(X_tr, y_train, X_te, d_model=FTT_D_MODEL,
                            n_heads=FTT_N_HEADS, ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                            dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                            lr=FTT_LR, weight_decay=FTT_WD, seed=seed),
                'A1': lambda: train_ft_transformer(X_tr_pf, y_train, X_te_pf, d_model=FTT_D_MODEL,
                            n_heads=FTT_N_HEADS, ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                            dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                            lr=FTT_LR, weight_decay=FTT_WD, seed=seed),
                'A2': lambda: train_rb_variant(X_tr, y_train, X_te, use_poly=False, seed=seed),
                'A3': lambda: train_rb_variant(X_tr, y_train, X_te, use_pca=False, seed=seed),
                'A4': lambda: train_rb_variant(X_tr, y_train, X_te, softmax_mixer=True, seed=seed),
                'A5': lambda: train_ft_transformer(X_tr, y_train, X_te, d_model=a5_d,
                            n_heads=4, ffn_dim=2*a5_d, n_blocks=FTT_N_BLOCKS,
                            dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                            lr=FTT_LR, weight_decay=FTT_WD, seed=seed),
                'RB': lambda: train_rb_variant(X_tr, y_train, X_te, seed=seed),
            }
            print(f"  Rep {r+1}/{N_REPEATS}:", end=" ", flush=True)
            for c in ABL:
                try:
                    y_pred, info = jobs[c]()
                    scores[c].append(r2_score(y_test, y_pred))
                    params[c] = info.get('n_params')
                except Exception as e:
                    print(f"[{c} ERROR: {e}]", end=" ")
                    scores[c].append(np.nan)
                print(c, end=" ", flush=True)
            print("| done")
        for c in ABL:
            print(f"  {c} [{LBL[c]}]  reps: {reps_str(scores[c])}  "
                  f"-> mean {np.nanmean(scores[c]):+.4f}  sd {np.nanstd(scores[c]):.4f}  "
                  f"params {params[c]:,}" if params[c] else "")
            ablation_rows.append(dict(dataset=ds_name, config=c, r2=np.nanmean(scores[c]),
                                      sd=np.nanstd(scores[c]), params=params[c],
                                      reps=";".join(f"{x:.4f}" for x in scores[c])))
        pd.DataFrame(ablation_rows).to_csv('ablation_grid_results.csv', index=False)

# ============================================================
# PART 2: UNCAPPED N (xZ71 W2/Q2)
# ============================================================
if RUN_UNCAPPED:
    print("\n" + "#" * 78 + "\n# PART 2: UNCAPPED N (California 20640, Kin8nm 8192, Protein 45730)\n" + "#" * 78)
    for ds_name in UNCAPPED:
        if ds_name not in datasets: continue
        X_use, y_use = datasets[ds_name]
        N_full, P = X_use.shape
        print(f"\n=== {ds_name} FULL N={N_full}, P={P} (models: OLS, RF, FT-T, RB) ===")
        scores = {m: [] for m in ['OLS', 'RF', 'FTT', 'RB']}
        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)
            sc = StandardScaler(); X_tr = sc.fit_transform(X_train); X_te = sc.transform(X_test)
            print(f"  Rep {r+1}/{N_REPEATS}:", end=" ", flush=True)
            try:
                m = LinearRegression().fit(X_tr, y_train)
                scores['OLS'].append(r2_score(y_test, m.predict(X_te)))
            except Exception as e: scores['OLS'].append(np.nan); print(f"[OLS:{e}]", end=" ")
            print("OLS", end=" ", flush=True)
            try:
                m = RandomForestRegressor(n_estimators=RF_TREES, max_features=RF_MTRY,
                                          n_jobs=-1, random_state=seed).fit(X_tr, y_train)
                scores['RF'].append(r2_score(y_test, m.predict(X_te)))
            except Exception as e: scores['RF'].append(np.nan); print(f"[RF:{e}]", end=" ")
            print("RF", end=" ", flush=True)
            try:
                y_pred, _ = train_ft_transformer(X_tr, y_train, X_te, d_model=FTT_D_MODEL,
                        n_heads=FTT_N_HEADS, ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                        dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                        lr=FTT_LR, weight_decay=FTT_WD, seed=seed)
                scores['FTT'].append(r2_score(y_test, y_pred))
            except Exception as e: scores['FTT'].append(np.nan); print(f"[FTT:{e}]", end=" ")
            print("FTT", end=" ", flush=True)
            try:
                y_pred, _ = train_rb_variant(X_tr, y_train, X_te, seed=seed)
                scores['RB'].append(r2_score(y_test, y_pred))
            except Exception as e: scores['RB'].append(np.nan); print(f"[RB:{e}]", end=" ")
            print("RB | done")
        for m in scores:
            print(f"  {m}  reps: {reps_str(scores[m])}  -> mean {np.nanmean(scores[m]):+.4f}  "
                  f"sd {np.nanstd(scores[m]):.4f}")
            uncapped_rows.append(dict(dataset=ds_name, N=N_full, model=m,
                                      r2=np.nanmean(scores[m]), sd=np.nanstd(scores[m]),
                                      reps=";".join(f"{x:.4f}" for x in scores[m])))
        pd.DataFrame(uncapped_rows).to_csv('uncapped_results.csv', index=False)

# ============================================================
# PART 3: TABPFN (Ayqz Q2)
# ============================================================
if RUN_TABPFN:
    print("\n" + "#" * 78 + "\n# PART 3: TABPFN REFERENCE (8 capped datasets)\n" + "#" * 78)
    try:
        try:
            from tabpfn import TabPFNRegressor
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn'])
            from tabpfn import TabPFNRegressor
        for ds_name, (X_full, y_full) in datasets.items():
            N_full, P = X_full.shape
            if N_full > MAX_N:
                np.random.seed(SEED)
                idx = np.random.choice(N_full, MAX_N, replace=False)
                X_use, y_use = X_full[idx], y_full[idx]
            else:
                X_use, y_use = X_full, y_full
            r2s = []
            print(f"  {ds_name} (N={len(y_use)}):", end=" ", flush=True)
            for r in range(N_REPEATS):
                seed = SEED + r * 1000
                X_train, X_test, y_train, y_test = train_test_split(
                    X_use, y_use, test_size=TEST_FRAC, random_state=seed)
                sc = StandardScaler(); X_tr = sc.fit_transform(X_train); X_te = sc.transform(X_test)
                try:
                    m = TabPFNRegressor(device='cuda' if torch.cuda.is_available() else 'cpu')
                    m.fit(X_tr, y_train)
                    r2s.append(r2_score(y_test, m.predict(X_te)))
                except Exception as e:
                    print(f"[rep{r+1}:{e}]", end=" ")
                    r2s.append(np.nan)
                print(f"{r2s[-1]:+.3f}" if not np.isnan(r2s[-1]) else "nan", end=" ", flush=True)
            print(f"-> mean {np.nanmean(r2s):+.4f} sd {np.nanstd(r2s):.4f}")
            tabpfn_rows.append(dict(dataset=ds_name, model='TabPFN', r2=np.nanmean(r2s),
                                    sd=np.nanstd(r2s), reps=";".join(f"{x:.4f}" for x in r2s)))
        pd.DataFrame(tabpfn_rows).to_csv('tabpfn_results.csv', index=False)
    except Exception as e:
        print(f"TabPFN unavailable, skipping (rebuttal falls back to 'added in revision'): {e}")

# ============================================================
# FINAL CONSOLIDATED REPORT
# ============================================================
print("\n" + "#" * 78 + f"\n# CONSOLIDATED REPORT ({(time.time()-T0)/60:.1f} min total)\n" + "#" * 78)

if ablation_rows:
    dfa = pd.DataFrame(ablation_rows)
    piv = dfa.pivot(index='dataset', columns='config', values='r2')[ABL]
    sdp = dfa.pivot(index='dataset', columns='config', values='sd')[ABL]
    print("\n--- Ablation grid: mean R2 (all cells) ---")
    print(piv.round(3).to_string())
    print("\n--- Ablation grid: sd across reps ---")
    print(sdp.round(3).to_string())
    print("\n--- Verification vs paper Table 1 (tol 0.02; Airfoil A0 by sign) ---")
    fails = 0
    for ds, tgt in PAPER.items():
        if ds not in piv.index: continue
        for c in ['A0', 'RB']:
            got = piv.loc[ds, c]
            ok = (got < 0) if (c == 'A0' and ds == 'Airfoil') else (abs(got - tgt[c]) <= 0.02)
            fails += (not ok)
            print(f"  {ds:<11} {c}: got {got:+.3f}  target {tgt[c]:+.3f}  {'OK' if ok else '** DRIFT **'}")
    print("VERIFIED: safe to quote." if fails == 0 else
          f"{fails} DRIFTED CELLS: reconcile before quoting new numbers.")
    ranks = piv.rank(axis=1, ascending=False)
    p8 = dfa[dfa.dataset == 'California'].set_index('config')['params']
    print("\n--- OpenReview markdown: ablation summary ---")
    print("| Config | Mean R2 | Avg rank | Wins vs FT-T | Params (P=8) |")
    print("|---|---|---|---|---|")
    for c in ABL:
        wins = int((piv[c] > piv['A0']).sum()) if c != 'A0' else '-'
        pc = f"{int(p8[c]):,}" if c in p8.index and pd.notna(p8[c]) else '-'
        print(f"| {LBL[c]} | {piv[c].mean():.3f} | {ranks[c].mean():.2f} | {wins} | {pc} |")
    print("\n--- OpenReview markdown: ablation per dataset ---")
    print("| Dataset | " + " | ".join(ABL) + " |")
    print("|---" * (len(ABL) + 1) + "|")
    for ds in piv.index:
        cells = " | ".join(f"{piv.loc[ds, c]:.3f}" if piv.loc[ds, c] > -1 else "<0" for c in ABL)
        print(f"| {ds} | {cells} |")
    print("\n--- LaTeX: ablation per dataset ---")
    print(r"\begin{tabular}{l" + "c" * len(ABL) + "}")
    print(r"\toprule")
    print("Dataset & " + " & ".join(ABL) + r" \\ \midrule")
    for ds in piv.index:
        cells = " & ".join(f"{piv.loc[ds, c]:.3f}" if piv.loc[ds, c] > -1 else "$<$0" for c in ABL)
        print(f"{ds} & {cells}" + r" \\")
    print(r"\bottomrule\end{tabular}")

if uncapped_rows:
    dfu = pd.DataFrame(uncapped_rows)
    print("\n--- Uncapped N: mean R2 ---")
    pivu = dfu.pivot(index='dataset', columns='model', values='r2')[['OLS', 'RF', 'FTT', 'RB']]
    print(pivu.round(3).to_string())
    print("\n--- OpenReview markdown: uncapped ---")
    print("| Dataset | N | OLS | RF | FT-T | Reg.Blk |")
    print("|---|---|---|---|---|---|")
    for ds in pivu.index:
        Nfull = int(dfu[dfu.dataset == ds]['N'].iloc[0])
        print(f"| {ds} | {Nfull} | " + " | ".join(f"{pivu.loc[ds, m]:.3f}"
              for m in ['OLS', 'RF', 'FTT', 'RB']) + " |")

if tabpfn_rows:
    dft = pd.DataFrame(tabpfn_rows)
    print("\n--- TabPFN reference column ---")
    for _, row in dft.iterrows():
        print(f"  {row['dataset']:<11} TabPFN R2 = {row['r2']:+.4f} (sd {row['sd']:.4f})")

print("\nSaved: ablation_grid_results.csv, uncapped_results.csv, tabpfn_results.csv")
print("Paste this cell's FULL output back into the claude.ai conversation.")

NameError: name 'POLY_PCA_COMP' is not defined

In [7]:
"""
============================================================
NeurIPS 31482 REBUTTAL EXPERIMENTS -- FULLY SELF-CONTAINED
Paste into ONE cell of a FRESH Colab notebook. GPU runtime.
No other cells needed. Runs everything and prints everything.
============================================================
PART 1  Ablation grid A0-A5+RB   PART 2  Uncapped N   PART 3  TabPFN
train_ft_transformer and load_datasets are verbatim from the v4
notebook that produced Table 1 (same seeds, splits, hyperparams).
"""

import numpy as np, pandas as pd, torch, torch.nn as nn, time, warnings, sys, subprocess
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')

# ================= CONFIG (identical to v4) =================
RUN_ABLATION = True
RUN_UNCAPPED = True
RUN_TABPFN   = True

N_REPEATS = 5
TEST_FRAC = 0.2
MAX_N = 5000
SEED = 42

RF_TREES = 500;  RF_MTRY = 1/3

FTT_D_MODEL = 64; FTT_N_HEADS = 4; FTT_FFN_DIM = 128; FTT_N_BLOCKS = 3
FTT_DROPOUT = 0.1; FTT_EPOCHS = 200; FTT_BS = 256; FTT_LR = 1e-3; FTT_WD = 1e-4

POLY_PCA_COMP = 200; POLY_D_MODEL = 64; POLY_FFN_DIM = 128
POLY_EPOCHS = 200; POLY_BS = 256; POLY_LR = 1e-3; POLY_DROPOUT = 0.1; POLY_WD = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PAPER = {
    'California': {'A0': 0.771, 'RB': 0.769}, 'Yacht':    {'A0': 0.445, 'RB': 0.960},
    'Energy':     {'A0': 0.992, 'RB': 0.995}, 'Concrete': {'A0': 0.652, 'RB': 0.892},
    'Airfoil':    {'A0': -75.451, 'RB': 0.927}, 'Abalone': {'A0': 0.314, 'RB': 0.331},
    'Kin8nm':     {'A0': 0.909, 'RB': 0.917}, 'Protein':  {'A0': 0.418, 'RB': 0.453},
}
UNCAPPED = ['California', 'Kin8nm', 'Protein']
ABL = ['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'RB']
LBL = {'A0': 'FT-Transformer (paper)', 'A1': 'FT-T + polynomial inputs',
       'A2': 'RB w/o polynomial expansion', 'A3': 'RB w/o PCA compression',
       'A4': 'RB skeleton, softmax instead of regression',
       'A5': 'FT-T parameter-matched (~30K)', 'RB': 'Regression Block (paper)'}

# ============================================================
# FT-TRANSFORMER  (verbatim from the v4 notebook)
# ============================================================
def train_ft_transformer(X_train, y_train, X_test,
                         d_model=64, n_heads=4, ffn_dim=128,
                         n_blocks=3, dropout=0.1,
                         epochs=50, batch_size=256, lr=1e-3,
                         weight_decay=1e-4, seed=42, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE
    nf = X_train.shape[1]; D = d_model

    class FTTransformer(nn.Module):
        def __init__(self, nf, d, nh, ffn, nblk, drop):
            super().__init__()
            self.feat_embeds = nn.ModuleList([nn.Linear(1, d) for _ in range(nf)])
            self.cls_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d, nhead=nh, dim_feedforward=ffn, dropout=drop,
                batch_first=True, activation='gelu')
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=nblk)
            self.norm = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)

        def forward(self, x):
            B = x.size(0)
            tokens = torch.stack([self.feat_embeds[i](x[:, i:i+1]) for i in range(x.size(1))], dim=1)
            tokens = torch.cat([self.cls_token.expand(B, -1, -1), tokens], dim=1)
            z = self.transformer(tokens)
            return self.head(self.norm(z[:, 0, :]))

    model = FTTransformer(nf, D, n_heads, ffn_dim, n_blocks, dropout).to(device)

    N = X_train.shape[0]
    n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)

    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    crit = nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    warmup_epochs = max(1, epochs // 10)
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return ep / warmup_epochs
        progress = (ep - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_val_loss = float('inf'); best_state = None
    patience_counter = 0; patience = 10
    for ep in range(1, epochs+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(Xt_val), yt_val).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test), 4096):
            xb = torch.from_numpy(X_test[i:i+4096].astype(np.float32)).to(device)
            preds.append(model(xb).cpu().numpy().ravel())
    return np.concatenate(preds), {'n_params': sum(p.numel() for p in model.parameters())}

# ============================================================
# RB VARIANTS (exact mirror of train_poly_model + flags)
# ============================================================
def train_rb_variant(X_train, y_train, X_test,
                     use_poly=True, use_pca=True, softmax_mixer=False,
                     n_components=POLY_PCA_COMP, d_model=POLY_D_MODEL,
                     ffn_dim=POLY_FFN_DIM, epochs=POLY_EPOCHS,
                     batch_size=POLY_BS, lr=POLY_LR, dropout=POLY_DROPOUT,
                     weight_decay=POLY_WD, n_heads=4, seed=42,
                     pca_fit_cap=5000):
    torch.manual_seed(seed); np.random.seed(seed)
    device = DEVICE; nf = X_train.shape[1]; D = d_model

    def poly_cross(x):
        B, S, Dm = x.shape
        per = torch.cat([x, x**2 - 1], dim=-1)
        pairs = torch.cat([x[:, i, :] * x[:, j, :] for i in range(S) for j in range(i, S)], dim=-1)
        return torch.cat([per, pairs.unsqueeze(1).expand(B, S, -1)], dim=-1)

    def features(x): return poly_cross(x) if use_poly else x

    class FrozenPCA(nn.Module):
        def __init__(self, mean, components):
            super().__init__()
            self.register_buffer('mean', torch.from_numpy(mean.astype(np.float32)))
            self.register_buffer('comp', torch.from_numpy(components.astype(np.float32)))
            self.out_dim = components.shape[0]
        def forward(self, x): return (x - self.mean) @ self.comp.T

    class Identity(nn.Module):
        def __init__(self, dim): super().__init__(); self.out_dim = dim
        def forward(self, x): return x

    class RBVariant(nn.Module):
        def __init__(self, nf, red_layer, d, ffn_dim, drop):
            super().__init__()
            self.embed = nn.Linear(1, d)
            self.pos = nn.Parameter(torch.randn(1, nf, d) * 0.02)
            if softmax_mixer:
                self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True, dropout=drop)
            else:
                self.red = red_layer
                self.W = nn.Linear(red_layer.out_dim, d)
            self.n1 = nn.LayerNorm(d)
            self.ffn = nn.Sequential(nn.Linear(d, ffn_dim), nn.ReLU(), nn.Dropout(drop),
                                     nn.Linear(ffn_dim, d), nn.Dropout(drop))
            self.n2 = nn.LayerNorm(d)
            self.head = nn.Linear(d, 1)
        def embed_x(self, x): return self.embed(x.unsqueeze(-1)) + self.pos
        def get_feat(self, emb): return self.red(features(emb))
        def forward(self, x):
            emb = self.embed_x(x)
            if softmax_mixer:
                a, _ = self.attn(emb, emb, emb)
                z = self.n1(emb + a)
            else:
                z = self.n1(emb + self.W(self.get_feat(emb)))
            return self.head(self.n2(z + self.ffn(z)).mean(1))

    red_layer = None
    if not softmax_mixer:
        torch.manual_seed(seed)
        tmp_embed = nn.Linear(1, D); tmp_pos = nn.Parameter(torch.randn(1, nf, D) * 0.02)
        if len(X_train) > pca_fit_cap:
            sub = np.random.RandomState(seed).choice(len(X_train), pca_fit_cap, replace=False)
            X_pca = X_train[sub]
        else:
            X_pca = X_train
        Xt = torch.from_numpy(X_pca.astype(np.float32))
        with torch.no_grad():
            emb = tmp_embed(Xt.unsqueeze(-1)) + tmp_pos
            phi_all = features(emb)
            feat_dim = phi_all.shape[-1]
            phi_all = phi_all.reshape(-1, feat_dim)
        if use_pca:
            n_comp = min(n_components, phi_all.shape[1], phi_all.shape[0])
            p = PCA(n_components=n_comp, random_state=seed); p.fit(phi_all.numpy())
            red_layer = FrozenPCA(p.mean_, p.components_)
        else:
            red_layer = Identity(feat_dim)

    torch.manual_seed(seed)
    model = RBVariant(nf, red_layer, D, ffn_dim, dropout).to(device)

    if not softmax_mixer:
        model.eval()
        WS_N = min(len(X_train), 20000)
        phis = []
        with torch.no_grad():
            for i in range(0, WS_N, 4096):
                xb = torch.from_numpy(X_train[i:i+4096].astype(np.float32)).to(device)
                phis.append(model.get_feat(model.embed_x(xb)).mean(1).cpu().numpy())
        phi_pooled = np.concatenate(phis)
        reg = Ridge(alpha=1.0); reg.fit(phi_pooled, y_train[:WS_N])
        with torch.no_grad():
            model.W.weight.data[0] = torch.from_numpy(reg.coef_.astype(np.float32)).to(device)
            model.W.bias.data.zero_()
            model.head.weight.data.zero_(); model.head.weight.data[0, 0] = 1.0
            model.head.bias.data.fill_(float(reg.intercept_))

    N = X_train.shape[0]; n_val = max(1, int(N * 0.15))
    idx = np.random.permutation(N)
    Xt_tr = torch.from_numpy(X_train[idx[n_val:]].astype(np.float32))
    yt_tr = torch.from_numpy(y_train[idx[n_val:]].astype(np.float32)).unsqueeze(1)
    Xt_val = torch.from_numpy(X_train[idx[:n_val]].astype(np.float32)).to(device)
    yt_val = torch.from_numpy(y_train[idx[:n_val]].astype(np.float32)).unsqueeze(1).to(device)
    tr_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.MSELoss()
    best, best_state, pat = float('inf'), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = crit(model(Xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xt_val), yt_val).item()
        if vl < best:
            best, best_state, pat = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= 10: break
    if best_state: model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test), 4096):
            xb = torch.from_numpy(X_test[i:i+4096].astype(np.float32)).to(device)
            preds.append(model(xb).cpu().numpy().ravel())
    return np.concatenate(preds), {'n_params': sum(p.numel() for p in model.parameters() if p.requires_grad)}

# ============================================================
# DATASETS (verbatim registry from v4)
# ============================================================
OPENML_REGISTRY = {
    'Concrete':  (44959,  (1000, 1100),  8,  'Concrete compressive strength'),
    'Energy':    (41478,  (700, 800),     8,  'Energy efficiency'),
    'Kin8nm':    (189,    (8000, 8300),   8,  'Kin8nm robot arm'),
    'Protein':   (44963,  (45000, 46000), 9,  'Protein structure CASP'),
    'Yacht':     (42370,  (300, 320),     6,  'Yacht hydrodynamics'),
    'Airfoil':   (44957,  (1400, 1600),   5,  'Airfoil self-noise'),
    'Abalone':   (183,    (4100, 4200),   7,  'Abalone age prediction'),
}
DS_ORDER = ['California', 'Yacht', 'Energy', 'Concrete', 'Airfoil', 'Abalone', 'Kin8nm', 'Protein']

def load_datasets():
    print(f"\nLoading {len(DS_ORDER)} datasets\n")
    loaded = {}
    try:
        d = fetch_california_housing()
        assert d.data.shape == (20640, 8)
        loaded['California'] = (d.data, d.target)
        print("  OK California: (20640, 8)")
    except Exception as e:
        print(f"  FAIL California: {e}")
    for name in DS_ORDER:
        if name == 'California': continue
        data_id, (n_min, n_max), expected_p, desc = OPENML_REGISTRY[name]
        try:
            data = fetch_openml(data_id=data_id, as_frame=True, parser='auto')
            X = data.data.select_dtypes(include=[np.number]).values
            y_raw = data.target
            if y_raw is None:
                raise ValueError("Target column not set in OpenML metadata")
            if hasattr(y_raw, 'shape') and len(y_raw.shape) > 1 and y_raw.shape[1] > 1:
                y_raw = y_raw.iloc[:, 0]
            elif hasattr(y_raw, 'columns'):
                y_raw = y_raw.iloc[:, 0]
            if hasattr(y_raw, 'cat') and y_raw.dtype.name == 'category':
                y = y_raw.cat.codes.values.astype(float)
            else:
                y = np.array(y_raw).astype(float)
            mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
            X, y = X[mask], y[mask]
            N, P = X.shape
            if not (n_min <= N <= n_max):
                print(f"  FAIL {name}: N={N} outside ({n_min}-{n_max})"); continue
            if P != expected_p:
                print(f"  FAIL {name}: P={P} != {expected_p}"); continue
            loaded[name] = (X, y)
            print(f"  OK {name}: ({N}, {P}) - {desc}")
        except Exception as e:
            print(f"  FAIL {name}: {e}")
    print(f"\n{len(loaded)}/{len(DS_ORDER)} datasets ready.\n")
    return {k: loaded[k] for k in DS_ORDER if k in loaded}

# ============================================================
# PARAM COUNTS + A5 SEARCH
# ============================================================
def ftt_param_count(nf, d, nh, ffn, nblk):
    m = nn.ModuleDict({
        'emb': nn.ModuleList([nn.Linear(1, d) for _ in range(nf)]),
        'enc': nn.TransformerEncoder(nn.TransformerEncoderLayer(
            d_model=d, nhead=nh, dim_feedforward=ffn, dropout=0.1,
            batch_first=True, activation='gelu'), num_layers=nblk),
        'norm': nn.LayerNorm(d), 'head': nn.Linear(d, 1)})
    return sum(p.numel() for p in m.parameters()) + d

def rb_param_count(nf, d=POLY_D_MODEL, ffn=POLY_FFN_DIM, ncomp=POLY_PCA_COMP):
    m = nn.ModuleDict({'e': nn.Linear(1, d), 'W': nn.Linear(ncomp, d),
                       'n1': nn.LayerNorm(d),
                       'f': nn.Sequential(nn.Linear(d, ffn), nn.Linear(ffn, d)),
                       'n2': nn.LayerNorm(d), 'h': nn.Linear(d, 1)})
    return sum(p.numel() for p in m.parameters()) + nf * d

def pick_a5_dmodel(nf, target):
    best_d, best_gap = None, float('inf')
    for d in [16, 20, 24, 28, 32, 36, 40, 48]:
        if d % 4: continue
        gap = abs(ftt_param_count(nf, d, 4, 2 * d, FTT_N_BLOCKS) - target)
        if gap < best_gap: best_d, best_gap = d, gap
    return best_d

def reps_str(v): return " ".join(f"{x:+.3f}" for x in v)

datasets = load_datasets()
ablation_rows, uncapped_rows, tabpfn_rows = [], [], []
T0 = time.time()

# ============================================================
# PART 1: ABLATION GRID
# ============================================================
if RUN_ABLATION:
    print("\n" + "#" * 78 + "\n# PART 1: ABLATION GRID (A0-A5 + RB)\n" + "#" * 78)
    for ds_name, (X_full, y_full) in datasets.items():
        N_full, P = X_full.shape
        if N_full > MAX_N:
            np.random.seed(SEED)
            idx = np.random.choice(N_full, MAX_N, replace=False)
            X_use, y_use = X_full[idx], y_full[idx]
        else:
            X_use, y_use = X_full, y_full
        rb_target = rb_param_count(P)
        a5_d = pick_a5_dmodel(P, rb_target)
        print(f"\n=== {ds_name} (N={len(y_use)}, P={P}) | RB params ~{rb_target:,} "
              f"-> A5 d_model={a5_d} (~{ftt_param_count(P, a5_d, 4, 2*a5_d, FTT_N_BLOCKS):,}) ===")
        scores = {c: [] for c in ABL}; params = {c: None for c in ABL}
        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)
            sc = StandardScaler(); X_tr = sc.fit_transform(X_train); X_te = sc.transform(X_test)
            pf = PolynomialFeatures(degree=2, include_bias=False)
            X_tr_pf = pf.fit_transform(X_tr); X_te_pf = pf.transform(X_te)
            sc2 = StandardScaler(); X_tr_pf = sc2.fit_transform(X_tr_pf); X_te_pf = sc2.transform(X_te_pf)
            jobs = {
                'A0': lambda: train_ft_transformer(X_tr, y_train, X_te, d_model=FTT_D_MODEL,
                            n_heads=FTT_N_HEADS, ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                            dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                            lr=FTT_LR, weight_decay=FTT_WD, seed=seed),
                'A1': lambda: train_ft_transformer(X_tr_pf, y_train, X_te_pf, d_model=FTT_D_MODEL,
                            n_heads=FTT_N_HEADS, ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                            dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                            lr=FTT_LR, weight_decay=FTT_WD, seed=seed),
                'A2': lambda: train_rb_variant(X_tr, y_train, X_te, use_poly=False, seed=seed),
                'A3': lambda: train_rb_variant(X_tr, y_train, X_te, use_pca=False, seed=seed),
                'A4': lambda: train_rb_variant(X_tr, y_train, X_te, softmax_mixer=True, seed=seed),
                'A5': lambda: train_ft_transformer(X_tr, y_train, X_te, d_model=a5_d,
                            n_heads=4, ffn_dim=2*a5_d, n_blocks=FTT_N_BLOCKS,
                            dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                            lr=FTT_LR, weight_decay=FTT_WD, seed=seed),
                'RB': lambda: train_rb_variant(X_tr, y_train, X_te, seed=seed),
            }
            print(f"  Rep {r+1}/{N_REPEATS}:", end=" ", flush=True)
            for c in ABL:
                try:
                    y_pred, info = jobs[c]()
                    scores[c].append(r2_score(y_test, y_pred))
                    params[c] = info.get('n_params')
                except Exception as e:
                    print(f"[{c} ERROR: {e}]", end=" ")
                    scores[c].append(np.nan)
                print(c, end=" ", flush=True)
            print("| done")
        for c in ABL:
            pstr = f"  params {params[c]:,}" if params[c] else ""
            print(f"  {c} [{LBL[c]}]  reps: {reps_str(scores[c])}  "
                  f"-> mean {np.nanmean(scores[c]):+.4f}  sd {np.nanstd(scores[c]):.4f}{pstr}")
            ablation_rows.append(dict(dataset=ds_name, config=c, r2=np.nanmean(scores[c]),
                                      sd=np.nanstd(scores[c]), params=params[c],
                                      reps=";".join(f"{x:.4f}" for x in scores[c])))
        pd.DataFrame(ablation_rows).to_csv('ablation_grid_results.csv', index=False)

# ============================================================
# PART 2: UNCAPPED N
# ============================================================
if RUN_UNCAPPED:
    print("\n" + "#" * 78 + "\n# PART 2: UNCAPPED N\n" + "#" * 78)
    for ds_name in UNCAPPED:
        if ds_name not in datasets: continue
        X_use, y_use = datasets[ds_name]
        N_full, P = X_use.shape
        print(f"\n=== {ds_name} FULL N={N_full}, P={P} (OLS, RF, FT-T, RB) ===")
        scores = {m: [] for m in ['OLS', 'RF', 'FTT', 'RB']}
        for r in range(N_REPEATS):
            seed = SEED + r * 1000
            X_train, X_test, y_train, y_test = train_test_split(
                X_use, y_use, test_size=TEST_FRAC, random_state=seed)
            sc = StandardScaler(); X_tr = sc.fit_transform(X_train); X_te = sc.transform(X_test)
            print(f"  Rep {r+1}/{N_REPEATS}:", end=" ", flush=True)
            try:
                m = LinearRegression().fit(X_tr, y_train)
                scores['OLS'].append(r2_score(y_test, m.predict(X_te)))
            except Exception as e: scores['OLS'].append(np.nan); print(f"[OLS:{e}]", end=" ")
            print("OLS", end=" ", flush=True)
            try:
                m = RandomForestRegressor(n_estimators=RF_TREES, max_features=RF_MTRY,
                                          n_jobs=-1, random_state=seed).fit(X_tr, y_train)
                scores['RF'].append(r2_score(y_test, m.predict(X_te)))
            except Exception as e: scores['RF'].append(np.nan); print(f"[RF:{e}]", end=" ")
            print("RF", end=" ", flush=True)
            try:
                y_pred, _ = train_ft_transformer(X_tr, y_train, X_te, d_model=FTT_D_MODEL,
                        n_heads=FTT_N_HEADS, ffn_dim=FTT_FFN_DIM, n_blocks=FTT_N_BLOCKS,
                        dropout=FTT_DROPOUT, epochs=FTT_EPOCHS, batch_size=FTT_BS,
                        lr=FTT_LR, weight_decay=FTT_WD, seed=seed)
                scores['FTT'].append(r2_score(y_test, y_pred))
            except Exception as e: scores['FTT'].append(np.nan); print(f"[FTT:{e}]", end=" ")
            print("FTT", end=" ", flush=True)
            try:
                y_pred, _ = train_rb_variant(X_tr, y_train, X_te, seed=seed)
                scores['RB'].append(r2_score(y_test, y_pred))
            except Exception as e: scores['RB'].append(np.nan); print(f"[RB:{e}]", end=" ")
            print("RB | done")
        for m in scores:
            print(f"  {m}  reps: {reps_str(scores[m])}  -> mean {np.nanmean(scores[m]):+.4f}  "
                  f"sd {np.nanstd(scores[m]):.4f}")
            uncapped_rows.append(dict(dataset=ds_name, N=N_full, model=m,
                                      r2=np.nanmean(scores[m]), sd=np.nanstd(scores[m]),
                                      reps=";".join(f"{x:.4f}" for x in scores[m])))
        pd.DataFrame(uncapped_rows).to_csv('uncapped_results.csv', index=False)

# ============================================================
# PART 3: TABPFN
# ============================================================
if RUN_TABPFN:
    print("\n" + "#" * 78 + "\n# PART 3: TABPFN REFERENCE\n" + "#" * 78)
    try:
        try:
            from tabpfn import TabPFNRegressor
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn'])
            from tabpfn import TabPFNRegressor
        for ds_name, (X_full, y_full) in datasets.items():
            N_full, P = X_full.shape
            if N_full > MAX_N:
                np.random.seed(SEED)
                idx = np.random.choice(N_full, MAX_N, replace=False)
                X_use, y_use = X_full[idx], y_full[idx]
            else:
                X_use, y_use = X_full, y_full
            r2s = []
            print(f"  {ds_name} (N={len(y_use)}):", end=" ", flush=True)
            for r in range(N_REPEATS):
                seed = SEED + r * 1000
                X_train, X_test, y_train, y_test = train_test_split(
                    X_use, y_use, test_size=TEST_FRAC, random_state=seed)
                sc = StandardScaler(); X_tr = sc.fit_transform(X_train); X_te = sc.transform(X_test)
                try:
                    m = TabPFNRegressor(device='cuda' if torch.cuda.is_available() else 'cpu')
                    m.fit(X_tr, y_train)
                    r2s.append(r2_score(y_test, m.predict(X_te)))
                except Exception as e:
                    print(f"[rep{r+1}:{e}]", end=" ")
                    r2s.append(np.nan)
                print(f"{r2s[-1]:+.3f}" if not np.isnan(r2s[-1]) else "nan", end=" ", flush=True)
            print(f"-> mean {np.nanmean(r2s):+.4f} sd {np.nanstd(r2s):.4f}")
            tabpfn_rows.append(dict(dataset=ds_name, model='TabPFN', r2=np.nanmean(r2s),
                                    sd=np.nanstd(r2s), reps=";".join(f"{x:.4f}" for x in r2s)))
        pd.DataFrame(tabpfn_rows).to_csv('tabpfn_results.csv', index=False)
    except Exception as e:
        print(f"TabPFN unavailable, skipping (rebuttal falls back to 'added in revision'): {e}")

# ============================================================
# FINAL CONSOLIDATED REPORT
# ============================================================
print("\n" + "#" * 78 + f"\n# CONSOLIDATED REPORT ({(time.time()-T0)/60:.1f} min total)\n" + "#" * 78)

if ablation_rows:
    dfa = pd.DataFrame(ablation_rows)
    piv = dfa.pivot(index='dataset', columns='config', values='r2').reindex(
        [d for d in DS_ORDER if d in dfa.dataset.unique()])[ABL]
    sdp = dfa.pivot(index='dataset', columns='config', values='sd').reindex(piv.index)[ABL]
    print("\n--- Ablation grid: mean R2 (all cells) ---")
    print(piv.round(3).to_string())
    print("\n--- Ablation grid: sd across reps ---")
    print(sdp.round(3).to_string())
    print("\n--- Verification vs paper Table 1 (tol 0.02; Airfoil A0 by sign) ---")
    fails = 0
    for ds, tgt in PAPER.items():
        if ds not in piv.index: continue
        for c in ['A0', 'RB']:
            got = piv.loc[ds, c]
            ok = (got < 0) if (c == 'A0' and ds == 'Airfoil') else (abs(got - tgt[c]) <= 0.02)
            fails += (not ok)
            print(f"  {ds:<11} {c}: got {got:+.3f}  target {tgt[c]:+.3f}  {'OK' if ok else '** DRIFT **'}")
    print("VERIFIED: safe to quote." if fails == 0 else
          f"{fails} DRIFTED CELLS: reconcile before quoting new numbers.")
    ranks = piv.rank(axis=1, ascending=False)
    p8 = dfa[dfa.dataset == 'California'].set_index('config')['params']
    print("\n--- OpenReview markdown: ablation summary ---")
    print("| Config | Mean R2 | Avg rank | Wins vs FT-T | Params (P=8) |")
    print("|---|---|---|---|---|")
    for c in ABL:
        wins = int((piv[c] > piv['A0']).sum()) if c != 'A0' else '-'
        pc = f"{int(p8[c]):,}" if c in p8.index and pd.notna(p8[c]) else '-'
        print(f"| {LBL[c]} | {piv[c].mean():.3f} | {ranks[c].mean():.2f} | {wins} | {pc} |")
    print("\n--- OpenReview markdown: ablation per dataset ---")
    print("| Dataset | " + " | ".join(ABL) + " |")
    print("|---" * (len(ABL) + 1) + "|")
    for ds in piv.index:
        cells = " | ".join(f"{piv.loc[ds, c]:.3f}" if piv.loc[ds, c] > -1 else "<0" for c in ABL)
        print(f"| {ds} | {cells} |")
    print("\n--- LaTeX: ablation per dataset ---")
    print(r"\begin{tabular}{l" + "c" * len(ABL) + "}")
    print(r"\toprule")
    print("Dataset & " + " & ".join(ABL) + r" \\ \midrule")
    for ds in piv.index:
        cells = " & ".join(f"{piv.loc[ds, c]:.3f}" if piv.loc[ds, c] > -1 else "$<$0" for c in ABL)
        print(f"{ds} & {cells}" + r" \\")
    print(r"\bottomrule\end{tabular}")

if uncapped_rows:
    dfu = pd.DataFrame(uncapped_rows)
    pivu = dfu.pivot(index='dataset', columns='model', values='r2')[['OLS', 'RF', 'FTT', 'RB']]
    print("\n--- Uncapped N: mean R2 ---")
    print(pivu.round(3).to_string())
    print("\n--- OpenReview markdown: uncapped ---")
    print("| Dataset | N | OLS | RF | FT-T | Reg.Blk |")
    print("|---|---|---|---|---|---|")
    for ds in pivu.index:
        Nfull = int(dfu[dfu.dataset == ds]['N'].iloc[0])
        print(f"| {ds} | {Nfull} | " + " | ".join(f"{pivu.loc[ds, m]:.3f}"
              for m in ['OLS', 'RF', 'FTT', 'RB']) + " |")

if tabpfn_rows:
    dft = pd.DataFrame(tabpfn_rows)
    print("\n--- TabPFN reference column ---")
    for _, row in dft.iterrows():
        print(f"  {row['dataset']:<11} TabPFN R2 = {row['r2']:+.4f} (sd {row['sd']:.4f})")

print("\nSaved: ablation_grid_results.csv, uncapped_results.csv, tabpfn_results.csv")
print("Paste this cell's FULL output back into the claude.ai conversation.")

Device: cuda
GPU: NVIDIA L4

Loading 8 datasets

  OK California: (20640, 8)
  OK Yacht: (308, 6) - Yacht hydrodynamics
  OK Energy: (768, 8) - Energy efficiency
  OK Concrete: (1030, 8) - Concrete compressive strength
  OK Airfoil: (1503, 5) - Airfoil self-noise
  OK Abalone: (4177, 7) - Abalone age prediction
  OK Kin8nm: (8192, 8) - Kin8nm robot arm
  OK Protein: (45730, 9) - Protein structure CASP

8/8 datasets ready.


##############################################################################
# PART 1: ABLATION GRID (A0-A5 + RB)
##############################################################################

=== California (N=5000, P=8) | RB params ~30,401 -> A5 d_model=36 (~33,013) ===
  Rep 1/5: A0 A1 A2 A3 A4 A5 RB | done
  Rep 2/5: A0 A1 A2 A3 A4 A5 RB | done
  Rep 3/5: A0 A1 A2 A3 A4 A5 RB | done
  Rep 4/5: A0 A1 A2 A3 A4 A5 RB | done
  Rep 5/5: A0 A1 A2 A3 A4 A5 RB | done
  A0 [FT-Transformer (paper)]  reps: +0.759 +0.783 +0.781 +0.751 +0.779  -> mean +0.7704  sd 0.0130  